# Graph Neural Network Approach FDiGNN - Analysing Simulation Results

# Dataset: Liver Graft biopsies Dataset (LGD)

FT-ICR-MS data obtained in Positive Ionization Mode. Samples are liver graft biopsies analysed at different points in time. A total fo 37 liver grafts were analysed (27 after donor brain death and 10 after cardiac death).

- 37 samples analysed while liver graft were in 'cold phase' storage after organ retrieval (Sample '21' outlier removed) - used as **'cold phase'**
- 37 samples analysed after liver graft transplant and stabilization of patient haemodynamic (Sample '21' outlier removed) - used as **'reperfusion'**
- 6 samples analysed during donor surgical phase (removed)
- 11 pooled Quality Control samples (removed)

#### Notebook Organization

- Reading LGD Formula-Difference Network and base functions


- Simulation result analysis for models from the 4-length Graphlets with no gaps simulations
- Simulation result analysis for models from the 5-length Graphlets with no gaps simulations
- Simulation result analysis for models from the 5-length Graphlets with neutral gaps simulations
- Simulation result analysis for models from the 5-length Graphlets with opposite gaps simulations
- Simulation result analysis for models from the 6-length Graphlets with no gaps simulations
- Simulation result analysis for models from the 6-length Graphlets with neutral gaps simulations
- Simulation result analysis for models from the 6-length Graphlets with opposite gaps simulations

#### Each Simulation Result Analysis Organization

- Setting Up the characteristics of the dataset and simulation
- Read the rank results for the metabolite importance measures of all simulations in that category for the pathway and single nodes in each simulation and join them in a DataFrame.Store results in a json, read the graphlet sets and read the results json.
- Plot figures to summarize the ranks of pathway and single (and when present gap nodes) across the simulations.
- Read all predictions of every single simulation/node/sample/quantile combinations and organize the results.
- Perform correlation analysis between centrality metrics and simulation analysis results.
- (For Simulations with gap nodes) Plot the gap node ranks distribution across simulation for the different models.

#### Metabolite Importances

- Gini Importance for RF model (RF)
- VIP Scores for PLS-DA model (PLS-DA)
- Prediction Impact Change for FDiGNN (FDiGNN)

# Needed Imports and Functions

In [ ]:
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import networkx as nx
import numpy as np
import pickle

In [ ]:
def process_results(filename, size):
    with open(filename) as f:
        graphlets = json.load(f)

    graphlets_dfs = {col: {} for col in graphlets['0'].keys()}
    for i in graphlets:
        for col in graphlets[i].keys():
            graphlets_dfs[col][i] = graphlets[i][col]
    for i in graphlets_dfs:
        graphlets_dfs[i] = pd.DataFrame(graphlets_dfs[i])
    graphlets_dfs.pop('Unnamed: 0')

    mannwhitney_pvalues = {}
    mannwhitney_pvalues_complete = pd.DataFrame()
    for key in graphlets_dfs:
        mannwhitney_pvalues[key] = stats.mannwhitneyu(graphlets_dfs[key].iloc[:size].values.flatten(),
                                                      graphlets_dfs[key].iloc[size:].values.flatten())[1]
        mannwhitney_pvalues_complete[key] = stats.mannwhitneyu(graphlets_dfs[key].iloc[:size],
                                                               graphlets_dfs[key].iloc[size:])[1]
    
    return graphlets, graphlets_dfs, mannwhitney_pvalues, mannwhitney_pvalues_complete

# LGD

In [ ]:
# load graph object from file
LGD_FDiN = pickle.load(open('LGD_FDiN_Final.pickle', 'rb'))
LGD_FDiN

In [ ]:
len(LGD_FDiN.edges())

## Graphlet Size 4 - No Gap

Reading and getting results

In [ ]:
# Parameters
size=4
ds_size = 1159
gnn_size = 350

colours = sns.color_palette('tab10', 20)

In [ ]:
# Obtaining results of each 
graphlets = {}
for i in range(20):
    graphlets[i] = pd.read_excel(f'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets4_Normal_{i}.xlsx').to_dict()

with open(f'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets4_Normal_TAG.json', 'w') as f:
    json.dump(graphlets, f)

In [ ]:
# Reading graphlets
with open('LGD_Simulations/Data/LGD_GNNPathwayTest_graphlets_size4.txt') as a:
    gs = a.read().split('\n')

with open('LGD_Simulations/Data/LGD_GNNPathwayTest_singlenodes_size4.txt') as a:
    sns_p = a.read().split('\n')

graphlets = []
for g in gs:
    graphlets.append(g.split(', '))

single_nodes = []
for g in sns_p:
    single_nodes.append(g.split(', '))

graphlets = graphlets[:-1]
single_nodes = single_nodes[:-1]

In [ ]:
graphlets_dict, graphlets_dfs, mannwhitney_pvalues, mannwhitney_pvalues_complete = process_results(
    'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets4_Normal_TAG.json', size)

cols = ['Pathway',] * size + ['Single',]*5
cols = cols*len(graphlets)

Setting up Figures

**For Supplementary Figure 4A.**

In [ ]:
%%capture --no-display
fig, axs = plt.subplots(2,3, figsize=(16,8), constrained_layout=True)
for ax, key in zip(axs.ravel(), ['RF - Normal', 'PLSDA - Normal', 'GNN - Normal - Pred',
                                'RF - Aleat.', 'PLSDA - Aleat.', 'GNN - Aleat. - Pred']):
    normal_ranks = graphlets_dfs[key].melt(ignore_index=False)
    normal_ranks['Type'] = cols
    normal_ranks.columns = ['DS', 'value', 'Type']
    normal_ranks['DS'] = 1 + normal_ranks['DS'].astype(int)
    sns.swarmplot(normal_ranks, x='DS', y='value', palette=colours, hue='Type', s=3,ax=ax, legend=False)
    ax.plot(range(len(graphlets)), graphlets_dfs[key].iloc[:size].median(), label='Pathway Nodes (Median)')
    ax.plot(range(len(graphlets)), graphlets_dfs[key].iloc[size:].median(), label='Single Nodes (Median)')
    ax.plot(range(len(graphlets)), graphlets_dfs[key].iloc[:size].max(), label='Pathway Nodes (Lowest)')
    ax.set_ylim([50,0])
    ax.set_xlabel('')
    ax.set_ylabel('')
    #ax.set_ylabel('Importance Rank', fontsize=20)
    if 'PLSDA' in key:
        t1 = 'PLS-DA'
    elif 'GNN' in key:
        t1 = 'FDiGNN'
    else:
        t1 = 'RF'
    if 'Normal' in key:
        t2 = ' - Original'
    else:
        t2 = ' - Randomized'
    mw_P = f' - {mannwhitney_pvalues[key]:.3e}'
    a, b = mw_P.split("e")
        
    ax.set_title(t1 + t2 + f'{a}x$10^{{{b}}}$', fontsize=18)
    for i in mannwhitney_pvalues_complete.index:
        if mannwhitney_pvalues_complete.loc[i, key] < 0.05:
            ax.text(i, 58, '*', fontsize=20, horizontalalignment='center')
axs[0][0].set_ylabel('Importance Rank', fontsize=15)
axs[1][0].set_ylabel('Importance Rank', fontsize=15)
#axs[1][1].set_xlabel('Graphlet + Single Node Sets', fontsize=14)
axs[0][0].legend(fontsize=15)
fig.supxlabel('Graphlet + Single Node Sets', fontsize=18)
#axs[2][1].set_xlabel('Dataset', fontsize=14)
plt.show()
fig.savefig('Paper_Figs/LGD_Graphlets4_TAG.png', dpi=600)
fig.savefig('Paper_Figs/LGD_Graphlets4_TAG.svg', dpi=600)

In [ ]:
%%capture --no-display
fig, (axl, axr) = plt.subplots(1,2, figsize=(16,6), constrained_layout=True)
a=0
colours_plot = sns.color_palette('tab20', 20)
for key in graphlets_dfs.keys():
    if 'Aleat.' in key:
        ax = axr
        a -= 2
    else:
        ax = axl
    path_ranks = graphlets_dfs[key].iloc[:size].apply(lambda x: x.sort_values().values)
    single_ranks = graphlets_dfs[key].iloc[size:].apply(lambda x: x.sort_values().values)
    
    single_ranks_melt = single_ranks.T.copy()
    single_ranks_melt.columns = range(1,6)
    single_ranks_melt = single_ranks_melt.melt(ignore_index=True)
    single_ranks_melt['Type'] = ['Single',] * len(single_ranks_melt)
    path_ranks_melt = path_ranks.T.copy()
    path_ranks_melt.columns = range(1,size+1)
    path_ranks_melt = path_ranks_melt.melt(ignore_index=True)
    path_ranks_melt['Type'] = ['Pathway',] * len(path_ranks_melt)
    normal_ranks = graphlets_dfs[key].melt(ignore_index=False)
    normal_ranks['Type'] = cols
    normal_ranks.columns = ['DS', 'value', 'Type']
   # sns.swarmplot(pd.concat((path_ranks_melt, single_ranks_melt)),
    #              x='variable', y='value', palette=colours, hue='Type', s=3,ax=ax)
    
    ax.plot(range(1, size+1), path_ranks.median(axis=1), label=key + ' - Path', c=colours_plot[a])
    #ax.fill_between(range(1, size+1), path_ranks.min(axis=1), path_ranks.max(axis=1), color=colours_plot[a], alpha=0.1)
    ax.plot(range(1, 5+1), single_ranks.median(axis=1), label=key + ' - Single', c=colours_plot[a+1])
    ax.set_ylim([0,30])
    #ax.set_xlabel('')
    a += 2
axr.legend(fontsize=15, bbox_to_anchor=(1,1))
axl.set_ylabel('Rank', fontsize=20)
axl.set_title('Normal Datasets', fontsize=20)
axr.set_title('Aleatorized Datasets', fontsize=20)
axl.set_xlabel('Best placed feature in the graphlet / single nodes', fontsize=16)
axr.set_xlabel('Best placed feature in the graphlet / single nodes', fontsize=16)
plt.show()

#### All Predictions

- Reading all predictions
- Obtaining the ranks of all nodes in each simulation
- See correlations and figures associating correlations

In [ ]:
# Reading Predictions
all_preds = {}
for i in range(20):
    with open(f'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets4_All_Predictions_TAG_{i}.json', 'r') as file:
        all_preds[i] = json.load(file)
all_preds.keys()

In [ ]:
# Get the prediction changes into a dictionary
all_effect = {'Normal + Path': {}, 'Aleatorized + Path': {}}
for key in all_effect:
    for l in range(20):
        all_effect[key][l] = {}
        out_normal = pd.DataFrame(all_preds[l][key]['Normal'])
        for node in all_preds[l][key]:
            if node != 'Normal':
                all_effect[key][l][node] = pd.DataFrame(columns=all_preds[l][key][node].keys())
                for q in all_preds[l][key][node]:
                    out_shuffled = pd.DataFrame(all_preds[l][key][node][q])
                    results = pd.DataFrame((out_normal.values - out_shuffled.values)).abs()
                    all_effect[key][l][node][q] = results[0]

In [ ]:
# Get the prediction impact of nodes per simulation
pred_changes = {}
for key in all_effect:
    pred_changes[key] = {}
    for i in all_effect[key]:
        new_df = pd.DataFrame(columns=range(72))
        for node in all_effect[key][i].keys():
            new_df.loc[node] = all_effect[key][i][node].max(axis=1).values#.sort_values().mean()
        new_df = (new_df/new_df.sum()).replace({np.nan:0})
        pred_changes[key][i] = new_df
# Get closeness centrality
LGD_closeness = pd.Series(nx.closeness_centrality(LGD_FDiN))

In [ ]:
# Calculate all correlations within each simulation and each dataset
all_ranks_normal = pd.DataFrame()
all_ranks_aleat = pd.DataFrame()
closeness = LGD_closeness.loc[list(LGD_FDiN.nodes())]
correlations_normal = pd.DataFrame(columns=['All', 'Graphlets', 'Single Nodes'])
correlations_aleat = pd.DataFrame(columns=['All', 'Graphlets', 'Single Nodes'])
for iteration in pred_changes['Normal + Path']:
    curr_df = pred_changes['Normal + Path'][iteration].median(axis=1).rank(ascending=False).loc[list(LGD_FDiN.nodes())]
    all_ranks_normal = pd.concat((all_ranks_normal, curr_df))
    all_corr = stats.spearmanr(curr_df, closeness)[0]
    grap_corr = stats.spearmanr(curr_df.loc[graphlets[iteration]], closeness.loc[graphlets[iteration]])[0]
    sn_corr = stats.spearmanr(curr_df.loc[single_nodes[iteration]], closeness.loc[single_nodes[iteration]])[0]
    correlations_normal.loc[iteration] = [all_corr, grap_corr, sn_corr]

    curr_df = pred_changes['Aleatorized + Path'][iteration].median(axis=1).rank(
        ascending=False).loc[list(LGD_FDiN.nodes())]
    all_ranks_aleat = pd.concat((all_ranks_aleat, curr_df))
    all_corr = stats.spearmanr(curr_df, closeness)[0]
    grap_corr = stats.spearmanr(curr_df.loc[graphlets[iteration]], closeness.loc[graphlets[iteration]])[0]
    sn_corr = stats.spearmanr(curr_df.loc[single_nodes[iteration]], closeness.loc[single_nodes[iteration]])[0]
    correlations_aleat.loc[iteration] = [all_corr, grap_corr, sn_corr]
all_closenesses = pd.concat([closeness,]*20)

In [ ]:
colors = []
sizes = []
for iteration in range(len(graphlets)):
    for i in LGD_FDiN.nodes():
        if i in graphlets[iteration]:
            colors.append('Red')
            sizes.append(5)
        elif i in single_nodes[iteration]:
            colors.append('Green')
            sizes.append(5)
        else:
            colors.append('skyblue')
            sizes.append(0.5)

In [ ]:
graphlet_corr = stats.spearmanr(all_ranks_normal.iloc[np.array(colors) == 'Red'],
                                all_closenesses.iloc[np.array(colors) == 'Red'])[0]

sn_corr = stats.spearmanr(all_ranks_normal.iloc[np.array(colors) == 'Green'],
                          all_closenesses.iloc[np.array(colors) == 'Green'])[0]

all_corr = stats.spearmanr(all_ranks_normal, all_closenesses)[0]

print('Original Dataset')
print(f'Graphlet     Correlation: {graphlet_corr}')
print(f'Single Nodes Correlation: {sn_corr}')
print(f'All          Correlation: {all_corr}')

In [ ]:
graphlet_corr = stats.spearmanr(all_ranks_aleat.iloc[np.array(colors) == 'Red'],
                                all_closenesses.iloc[np.array(colors) == 'Red'])[0]

sn_corr = stats.spearmanr(all_ranks_aleat.iloc[np.array(colors) == 'Green'],
                          all_closenesses.iloc[np.array(colors) == 'Green'])[0]

all_corr = stats.spearmanr(all_ranks_aleat, all_closenesses)[0]

print('Randomized Dataset')
print(f'Graphlet     Correlation: {graphlet_corr}')
print(f'Single Nodes Correlation: {sn_corr}')
print(f'All          Correlation: {all_corr}')

Correlations Figure

In [ ]:
fig, (axl,axr) = plt.subplots(1,2, figsize=(12,4), constrained_layout=True)
axl.scatter(all_ranks_normal, all_closenesses, s=sizes, c=colors)
axr.scatter(all_ranks_aleat, all_closenesses, s=sizes, c=colors)

## Graphlet Size 5 - No Gap

Reading and getting results

In [ ]:
# Parameters
size=5
ds_size = 1159
gnn_size = 350

colours = sns.color_palette('tab10', 20)

In [ ]:
# Obtaining results of each
graphlets = {}
for i in range(20):
    graphlets[i] = pd.read_excel(f'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets5_Normal_{i}.xlsx').to_dict()

with open(f'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets5_Normal_TAG.json', 'w') as f:
    json.dump(graphlets, f)

In [ ]:
# Reading graphlets
with open('LGD_Simulations/Data/LGD_GNNPathwayTest_graphlets_size5.txt') as a:
    gs = a.read().split('\n')

with open('LGD_Simulations/Data/LGD_GNNPathwayTest_singlenodes_size5.txt') as a:
    sns_p = a.read().split('\n')

graphlets = []
for g in gs:
    graphlets.append(g.split(', '))

single_nodes = []
for g in sns_p:
    single_nodes.append(g.split(', '))

graphlets = graphlets[:-1]
single_nodes = single_nodes[:-1]

with open('LGD_Simulations/Data/LGD_5Graphlets_gaps.txt') as a:
    gap_nodes = a.read().split('\n')[:-1]

In [ ]:
graphlets_dict, graphlets_dfs, mannwhitney_pvalues, mannwhitney_pvalues_complete = process_results(
    'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets5_Normal_TAG.json', size)

cols = ['Pathway',] * size + ['Single',]*5
cols = cols*len(graphlets)

Setting up Figures

**For Figure 5A.**

In [ ]:
fig, (axl, axr) = plt.subplots(1,2, figsize=(5,6), constrained_layout=True)

normal_ranks = pd.DataFrame()
for key in ['RF - Normal', 'PLSDA - Normal', 'GNN - Normal - Pred']:
    normal_ranks2 = graphlets_dfs[key].melt(ignore_index=False)
    normal_ranks2['Type'] = cols
    normal_ranks2.columns = ['DS', 'value', 'Type']
    if 'PLSDA' in key:
        t1 = 'PLS-DA'
    elif 'GNN' in key:
        t1 = 'FDiGNN'
    else:
        t1 = 'RF'
    if mannwhitney_pvalues[key] < 0.001:
        t1 += '\n***'
    elif mannwhitney_pvalues[key] < 0.01:
        t1 += '\n**'
    elif mannwhitney_pvalues[key] < 0.05:
        t1 += '\n*'
    else:
        t1 += '\n '
    normal_ranks2['DS'] = t1
    normal_ranks = pd.concat((normal_ranks, normal_ranks2))

sns.boxplot(data=normal_ranks, x="DS", y="value", hue="Type", linewidth=1, ax=axl, fill=True, gap=0.1)
axl.set_title('Original', fontsize=13)
axl.set_xlabel('')

normal_ranks = pd.DataFrame()
for key in ['RF - Aleat.', 'PLSDA - Aleat.', 'GNN - Aleat. - Pred']:
    normal_ranks2 = graphlets_dfs[key].melt(ignore_index=False)
    normal_ranks2['Type'] = cols
    normal_ranks2.columns = ['DS', 'value', 'Type']
    if 'PLSDA' in key:
        t1 = 'PLS-DA'
    elif 'GNN' in key:
        t1 = 'FDiGNN'
    else:
        t1 = 'RF'
    if mannwhitney_pvalues[key] < 0.001:
        t1 += '\n***'
    elif mannwhitney_pvalues[key] < 0.01:
        t1 += '\n**'
    elif mannwhitney_pvalues[key] < 0.05:
        t1 += '\n*'
    else:
        t1 += '\n '
    normal_ranks2['DS'] = t1
    normal_ranks = pd.concat((normal_ranks, normal_ranks2))
    
bxp = sns.boxplot(data=normal_ranks, x="DS", y="value", hue="Type", linewidth=1, ax=axr, fill=True, gap=0.1)
axr.set_title('Randomized', fontsize=13)
axr.set_xlabel('')
axr.set_ylabel('')

axl.set_ylabel('Importance Rank', fontsize=13)

axl.tick_params(axis='x', labelbottom=False, labeltop=True, labelsize=11)
axl.xaxis.set_ticks_position('top')
axr.tick_params(axis='x', labelbottom=False, labeltop=True, labelsize=11)
axr.xaxis.set_ticks_position('top')
axl.legend(fontsize=12)
axr.legend().set_visible(False)
axl.set_ylim([100,0])
axr.set_ylim([100,0])
axl.set_yticks(range(0,101, 10))
axr.set_yticks(range(0,101, 10))
plt.suptitle('       LGD (5 node graphlets)', fontsize=15)
plt.show()
fig.savefig('Boxplot_LGD_5Graphlet_NoGap.svg', dpi=400)

**For Supplementary Figure 3A.**

In [ ]:
%%capture --no-display
fig, axs = plt.subplots(2,3, figsize=(16,8), constrained_layout=True)
for ax, key in zip(axs.ravel(), ['RF - Normal', 'PLSDA - Normal', 'GNN - Normal - Pred',
                                'RF - Aleat.', 'PLSDA - Aleat.', 'GNN - Aleat. - Pred']):
    normal_ranks = graphlets_dfs[key].melt(ignore_index=False)
    normal_ranks['Type'] = cols
    normal_ranks.columns = ['DS', 'value', 'Type']
    normal_ranks['DS'] = 1 + normal_ranks['DS'].astype(int)
    sns.swarmplot(normal_ranks, x='DS', y='value', palette=colours, hue='Type', s=3,ax=ax, legend=False)
    ax.plot(range(len(graphlets)), graphlets_dfs[key].iloc[:size].median(), label='Pathway Nodes (Median)')
    ax.plot(range(len(graphlets)), graphlets_dfs[key].iloc[size:].median(), label='Single Nodes (Median)')
    ax.plot(range(len(graphlets)), graphlets_dfs[key].iloc[:size].max(), label='Pathway Nodes (Lowest)')
    ax.set_ylim([50,0])
    ax.set_xlabel('')
    ax.set_ylabel('')
    #ax.set_ylabel('Importance Rank', fontsize=20)
    if 'PLSDA' in key:
        t1 = 'PLS-DA'
    elif 'GNN' in key:
        t1 = 'FDiGNN'
    else:
        t1 = 'RF'
    if 'Normal' in key:
        t2 = ' - Original'
    else:
        t2 = ' - Randomized'
        
    mw_P = f' - {mannwhitney_pvalues[key]:.3e}'
    a, b = mw_P.split("e")
        
    ax.set_title(t1 + t2 + f'{a}x$10^{{{b}}}$', fontsize=18)
    for i in mannwhitney_pvalues_complete.index:
        if mannwhitney_pvalues_complete.loc[i, key] < 0.05:
            ax.text(i, 58, '*', fontsize=20, horizontalalignment='center')
axs[0][0].set_ylabel('Importance Rank', fontsize=15)
axs[1][0].set_ylabel('Importance Rank', fontsize=15)
#axs[1][1].set_xlabel('Graphlet + Single Node Sets', fontsize=14)
axs[0][0].legend(fontsize=15)
fig.supxlabel('Graphlet + Single Node Sets', fontsize=18)
#axs[2][1].set_xlabel('Dataset', fontsize=14)
plt.show()
fig.savefig('Paper_Figs/LGD_Graphlets5_TAG.png', dpi=600)
fig.savefig('Paper_Figs/LGD_Graphlets5_TAG.svg', dpi=600)

In [ ]:
%%capture --no-display
fig, (axl, axr) = plt.subplots(1,2, figsize=(16,6), constrained_layout=True)
a=0
colours_plot = sns.color_palette('tab20', 20)
for key in graphlets_dfs.keys():
    if 'Aleat.' in key:
        ax = axr
        a -= 2
    else:
        ax = axl
    path_ranks = graphlets_dfs[key].iloc[:size].apply(lambda x: x.sort_values().values)
    single_ranks = graphlets_dfs[key].iloc[size:].apply(lambda x: x.sort_values().values)
    
    single_ranks_melt = single_ranks.T.copy()
    single_ranks_melt.columns = range(1,6)
    single_ranks_melt = single_ranks_melt.melt(ignore_index=True)
    single_ranks_melt['Type'] = ['Single',] * len(single_ranks_melt)
    path_ranks_melt = path_ranks.T.copy()
    path_ranks_melt.columns = range(1,size+1)
    path_ranks_melt = path_ranks_melt.melt(ignore_index=True)
    path_ranks_melt['Type'] = ['Pathway',] * len(path_ranks_melt)
    normal_ranks = graphlets_dfs[key].melt(ignore_index=False)
    normal_ranks['Type'] = cols
    normal_ranks.columns = ['DS', 'value', 'Type']
   # sns.swarmplot(pd.concat((path_ranks_melt, single_ranks_melt)),
    #              x='variable', y='value', palette=colours, hue='Type', s=3,ax=ax)
    
    ax.plot(range(1, size+1), path_ranks.median(axis=1), label=key + ' - Path', c=colours_plot[a])
    #ax.fill_between(range(1, size+1), path_ranks.min(axis=1), path_ranks.max(axis=1), color=colours_plot[a], alpha=0.1)
    ax.plot(range(1, 5+1), single_ranks.median(axis=1), label=key + ' - Single', c=colours_plot[a+1])
    ax.set_ylim([0,30])
    #ax.set_xlabel('')
    a += 2
axr.legend(fontsize=15, bbox_to_anchor=(1,1))
axl.set_ylabel('Rank', fontsize=20)
axl.set_title('Normal Datasets', fontsize=20)
axr.set_title('Aleatorized Datasets', fontsize=20)
axl.set_xlabel('Best placed feature in the graphlet / single nodes', fontsize=16)
axr.set_xlabel('Best placed feature in the graphlet / single nodes', fontsize=16)
plt.show()

#### All Predictions

- Reading all predictions
- Obtaining the ranks of all nodes in each simulation
- See correlations and figures associating correlations

In [ ]:
# Reading Predictions
all_preds = {}
for i in range(20):
    with open(f'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets5_All_Predictions_TAG_{i}.json', 'r') as file:
        all_preds[i] = json.load(file)
all_preds.keys()

In [ ]:
# Get the prediction changes into a dictionary
all_effect = {'Normal + Path': {}, 'Aleatorized + Path': {}}
for key in all_effect:
    for l in range(20):
        all_effect[key][l] = {}
        out_normal = pd.DataFrame(all_preds[l][key]['Normal'])
        for node in all_preds[l][key]:
            if node != 'Normal':
                all_effect[key][l][node] = pd.DataFrame(columns=all_preds[l][key][node].keys())
                for q in all_preds[l][key][node]:
                    out_shuffled = pd.DataFrame(all_preds[l][key][node][q])
                    results = pd.DataFrame((out_normal.values - out_shuffled.values)).abs()
                    all_effect[key][l][node][q] = results[0]

In [ ]:
# Get the prediction impact of nodes per simulation
pred_changes = {}
for key in all_effect:
    pred_changes[key] = {}
    for i in all_effect[key]:
        new_df = pd.DataFrame(columns=range(72))
        for node in all_effect[key][i].keys():
            new_df.loc[node] = all_effect[key][i][node].max(axis=1).values#.sort_values().mean()
        new_df = (new_df/new_df.sum()).replace({np.nan:0})
        pred_changes[key][i] = new_df
# Get closeness centrality
LGD_closeness = pd.Series(nx.closeness_centrality(LGD_FDiN))

In [ ]:
# Calculate all correlations within each simulation and each dataset
all_ranks_normal = pd.DataFrame()
all_ranks_aleat = pd.DataFrame()
closeness = LGD_closeness.loc[list(LGD_FDiN.nodes())]
correlations_normal = pd.DataFrame(columns=['All', 'Graphlets', 'Single Nodes'])
correlations_aleat = pd.DataFrame(columns=['All', 'Graphlets', 'Single Nodes'])
for iteration in pred_changes['Normal + Path']:
    curr_df = pred_changes['Normal + Path'][iteration].median(axis=1).rank(ascending=False).loc[list(LGD_FDiN.nodes())]
    all_ranks_normal = pd.concat((all_ranks_normal, curr_df))
    all_corr = stats.spearmanr(curr_df, closeness)[0]
    grap_corr = stats.spearmanr(curr_df.loc[graphlets[iteration]], closeness.loc[graphlets[iteration]])[0]
    sn_corr = stats.spearmanr(curr_df.loc[single_nodes[iteration]], closeness.loc[single_nodes[iteration]])[0]
    correlations_normal.loc[iteration] = [all_corr, grap_corr, sn_corr]

    curr_df = pred_changes['Aleatorized + Path'][iteration].median(axis=1).rank(
        ascending=False).loc[list(LGD_FDiN.nodes())]
    all_ranks_aleat = pd.concat((all_ranks_aleat, curr_df))
    all_corr = stats.spearmanr(curr_df, closeness)[0]
    grap_corr = stats.spearmanr(curr_df.loc[graphlets[iteration]], closeness.loc[graphlets[iteration]])[0]
    sn_corr = stats.spearmanr(curr_df.loc[single_nodes[iteration]], closeness.loc[single_nodes[iteration]])[0]
    correlations_aleat.loc[iteration] = [all_corr, grap_corr, sn_corr]
all_closenesses = pd.concat([closeness,]*20)

In [ ]:
colors = []
sizes = []
for iteration in range(len(graphlets)):
    for i in LGD_FDiN.nodes():
        if i in graphlets[iteration]:
            colors.append('Red')
            sizes.append(5)
        elif i in single_nodes[iteration]:
            colors.append('Green')
            sizes.append(5)
        else:
            colors.append('skyblue')
            sizes.append(0.5)

In [ ]:
graphlet_corr = stats.spearmanr(all_ranks_normal.iloc[np.array(colors) == 'Red'],
                                all_closenesses.iloc[np.array(colors) == 'Red'])[0]

sn_corr = stats.spearmanr(all_ranks_normal.iloc[np.array(colors) == 'Green'],
                          all_closenesses.iloc[np.array(colors) == 'Green'])[0]

all_corr = stats.spearmanr(all_ranks_normal, all_closenesses)[0]

print('Original Dataset')
print(f'Graphlet     Correlation: {graphlet_corr}')
print(f'Single Nodes Correlation: {sn_corr}')
print(f'All          Correlation: {all_corr}')

In [ ]:
graphlet_corr = stats.spearmanr(all_ranks_aleat.iloc[np.array(colors) == 'Red'],
                                all_closenesses.iloc[np.array(colors) == 'Red'])[0]

sn_corr = stats.spearmanr(all_ranks_aleat.iloc[np.array(colors) == 'Green'],
                          all_closenesses.iloc[np.array(colors) == 'Green'])[0]

all_corr = stats.spearmanr(all_ranks_aleat, all_closenesses)[0]

print('Randomized Dataset')
print(f'Graphlet     Correlation: {graphlet_corr}')
print(f'Single Nodes Correlation: {sn_corr}')
print(f'All          Correlation: {all_corr}')

Correlations Figure

In [ ]:
fig, (axl,axr) = plt.subplots(1,2, figsize=(12,4), constrained_layout=True)
axl.scatter(all_ranks_normal, all_closenesses, s=sizes, c=colors)
axr.scatter(all_ranks_aleat, all_closenesses, s=sizes, c=colors)

## Graphlet Size 5 - Gap

Reading and getting results

In [ ]:
# Parameters
size=5
ds_size = 1159
gnn_size = 350

colours = sns.color_palette('tab10', 20)

In [ ]:
# Obtaining results of each
graphlets = {}
for i in range(20):
    graphlets[i] = pd.read_excel(f'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets5Gap_Normal_{i}.xlsx').to_dict()

with open(f'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets5_Gap_TAG.json', 'w') as f:
    json.dump(graphlets, f)

In [ ]:
# Reading graphlets
with open('LGD_Simulations/Data/LGD_GNNPathwayTest_graphlets_size5.txt') as a:
    gs = a.read().split('\n')

with open('LGD_Simulations/Data/LGD_GNNPathwayTest_singlenodes_size5.txt') as a:
    sns_p = a.read().split('\n')

graphlets = []
for g in gs:
    graphlets.append(g.split(', '))

single_nodes = []
for g in sns_p:
    single_nodes.append(g.split(', '))

graphlets = graphlets[:-1]
single_nodes = single_nodes[:-1]

with open('LGD_Simulations/Data/LGD_5Graphlets_gaps.txt') as a:
    gap_nodes = a.read().split('\n')[:-1]

In [ ]:
graphlets_dict, graphlets_dfs, mannwhitney_pvalues, mannwhitney_pvalues_complete = process_results(
    'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets5_Gap_TAG.json', size)

cols = []
for g in range(len(graphlets)):
    g_g = graphlets[g]
    special = gap_nodes[g]
    for node in g_g:
        if node == special:
            cols.append('Gap')
        else:
            cols.append('Pathway')
    cols.extend(['Single', 'Single','Single', 'Single', 'Single'])

Setting up Figures

**For Figure 6A.**

In [ ]:
fig, (axl, axr) = plt.subplots(1,2, figsize=(5,6), constrained_layout=True)

normal_ranks = pd.DataFrame()
for key in ['RF - Normal', 'PLSDA - Normal', 'GNN - Normal - Pred']:
    normal_ranks2 = graphlets_dfs[key].melt(ignore_index=False)
    normal_ranks2['Type'] = cols
    normal_ranks2.columns = ['DS', 'value', 'Type']
    if 'PLSDA' in key:
        t1 = 'PLS-DA'
    elif 'GNN' in key:
        t1 = 'FDiGNN'
    else:
        t1 = 'RF'
    if mannwhitney_pvalues[key] < 0.001:
        t1 += '\n***'
    elif mannwhitney_pvalues[key] < 0.01:
        t1 += '\n**'
    elif mannwhitney_pvalues[key] < 0.05:
        t1 += '\n*'
    else:
        t1 += '\n '
    normal_ranks2['DS'] = t1
    normal_ranks = pd.concat((normal_ranks, normal_ranks2))

sns.boxplot(data=normal_ranks, x="DS", y="value", hue="Type", linewidth=1, ax=axl, fill=True, gap=0.1,
           hue_order=['Pathway', 'Single', 'Gap'])
axl.set_title('Original', fontsize=13)
axl.set_xlabel('')

normal_ranks = pd.DataFrame()
for key in ['RF - Aleat.', 'PLSDA - Aleat.', 'GNN - Aleat. - Pred']:
    normal_ranks2 = graphlets_dfs[key].melt(ignore_index=False)
    normal_ranks2['Type'] = cols
    normal_ranks2.columns = ['DS', 'value', 'Type']
    if 'PLSDA' in key:
        t1 = 'PLS-DA'
    elif 'GNN' in key:
        t1 = 'FDiGNN'
    else:
        t1 = 'RF'
    if mannwhitney_pvalues[key] < 0.001:
        t1 += '\n***'
    elif mannwhitney_pvalues[key] < 0.01:
        t1 += '\n**'
    elif mannwhitney_pvalues[key] < 0.05:
        t1 += '\n*'
    else:
        t1 += '\n '
    normal_ranks2['DS'] = t1
    normal_ranks = pd.concat((normal_ranks, normal_ranks2))
    
bxp = sns.boxplot(data=normal_ranks, x="DS", y="value", hue="Type", linewidth=1, ax=axr, fill=True, gap=0.1,
                 hue_order=['Pathway', 'Single', 'Gap'])
axr.set_title('Randomized', fontsize=13)
axr.set_xlabel('')
axr.set_ylabel('')

axl.set_ylabel('Importance Rank', fontsize=13)

axl.tick_params(axis='x', labelbottom=False,labeltop=True, labelsize=11)
axl.xaxis.set_ticks_position('top')
axr.tick_params(axis='x', labelbottom=False,labeltop=True, labelsize=11)
axr.xaxis.set_ticks_position('top')
axl.legend(fontsize=12)
axr.legend().set_visible(False)
axl.set_ylim([100,0])
axr.set_ylim([100,0])
axl.set_yticks(range(0,101, 10))
axr.set_yticks(range(0,101, 10))
plt.suptitle('       LGD (5 node graphlets with Neutral Gap)', fontsize=15)
plt.show()
fig.savefig('Boxplot_LGD_5Graphlet_Gap.svg', dpi=400)

**For Supplementary Figure 6A.**

In [ ]:
%%capture --no-display
fig, axs = plt.subplots(2,3, figsize=(16,8), constrained_layout=True)
gap_ranks = pd.DataFrame()
for ax, key in zip(axs.ravel(), ['RF - Normal', 'PLSDA - Normal', 'GNN - Normal - Pred',
                                'RF - Aleat.', 'PLSDA - Aleat.', 'GNN - Aleat. - Pred']):
    normal_ranks = graphlets_dfs[key].melt(ignore_index=False)
    normal_ranks['Type'] = cols
    normal_ranks.columns = ['DS', 'value', 'Type']
    normal_ranks['DS'] = 1 + normal_ranks['DS'].astype(int)
    gap_ranks[key] = normal_ranks[normal_ranks['Type'] == 'Gap']['value'].values
    sns.swarmplot(normal_ranks, x='DS', y='value', palette=colours, hue='Type', s=3,ax=ax,
                  hue_order=['Pathway', 'Single', 'Gap'], legend=False)
    ax.plot(range(len(graphlets)), graphlets_dfs[key].iloc[:size].median(), label='Pathway Nodes (Median)')
    ax.plot(range(len(graphlets)), graphlets_dfs[key].iloc[size:].median(), label='Single Nodes (Median)')
    ax.plot(range(len(graphlets)), normal_ranks[normal_ranks['Type'] == 'Gap']['value'].values,
            label='Gap Nodes')
    ax.set_ylim([50,0])
    ax.set_xlabel('')
    ax.set_ylabel('')
    #ax.set_ylabel('Importance Rank', fontsize=20)
    if 'PLSDA' in key:
        t1 = 'PLS-DA'
    elif 'GNN' in key:
        t1 = 'FDiGNN'
    else:
        t1 = 'RF'
    if 'Normal' in key:
        t2 = ' - Original'
    else:
        t2 = ' - Randomized'
        
    mw_P = f' - {mannwhitney_pvalues[key]:.3e}'
    a, b = mw_P.split("e")
        
    ax.set_title(t1 + t2 + f'{a}x$10^{{{b}}}$', fontsize=18)
    for i in mannwhitney_pvalues_complete.index:
        if mannwhitney_pvalues_complete.loc[i, key] < 0.05:
            ax.text(i, 58, '*', fontsize=20, horizontalalignment='center')
axs[0][0].set_ylabel('Importance Rank', fontsize=15)
axs[1][0].set_ylabel('Importance Rank', fontsize=15)
#axs[1][1].set_xlabel('Graphlet + Single Node Sets', fontsize=14)
axs[0][0].legend(fontsize=15)
fig.supxlabel('Graphlet + Single Node Sets', fontsize=18)
#axs[2][1].set_xlabel('Dataset', fontsize=14)
plt.show()
fig.savefig('Paper_Figs/LGD_Graphlets5_Gap_TAG.png', dpi=600)
fig.savefig('Paper_Figs/LGD_Graphlets5_Gap_TAG.svg', dpi=600)

In [ ]:
%%capture --no-display
fig, (axl, axr) = plt.subplots(1,2, figsize=(16,6), constrained_layout=True)
a=0
colours_plot = sns.color_palette('tab20', 20)
for key in graphlets_dfs.keys():
    if 'Aleat.' in key:
        ax = axr
        a -= 2
    else:
        ax = axl
    path_ranks = graphlets_dfs[key].iloc[:size].apply(lambda x: x.sort_values().values)
    single_ranks = graphlets_dfs[key].iloc[size:].apply(lambda x: x.sort_values().values)
    
    single_ranks_melt = single_ranks.T.copy()
    single_ranks_melt.columns = range(1,6)
    single_ranks_melt = single_ranks_melt.melt(ignore_index=True)
    single_ranks_melt['Type'] = ['Single',] * len(single_ranks_melt)
    path_ranks_melt = path_ranks.T.copy()
    path_ranks_melt.columns = range(1,size+1)
    path_ranks_melt = path_ranks_melt.melt(ignore_index=True)
    path_ranks_melt['Type'] = ['Pathway',] * len(path_ranks_melt)
    normal_ranks = graphlets_dfs[key].melt(ignore_index=False)
    normal_ranks['Type'] = cols
    normal_ranks.columns = ['DS', 'value', 'Type']
   # sns.swarmplot(pd.concat((path_ranks_melt, single_ranks_melt)),
    #              x='variable', y='value', palette=colours, hue='Type', s=3,ax=ax)
    
    ax.plot(range(1, size+1), path_ranks.median(axis=1), label=key + ' - Path', c=colours_plot[a])
    #ax.fill_between(range(1, size+1), path_ranks.min(axis=1), path_ranks.max(axis=1), color=colours_plot[a], alpha=0.1)
    ax.plot(range(1, 5+1), single_ranks.median(axis=1), label=key + ' - Single', c=colours_plot[a+1])
    ax.set_ylim([0,30])
    #ax.set_xlabel('')
    a += 2
axr.legend(fontsize=15, bbox_to_anchor=(1,1))
axl.set_ylabel('Rank', fontsize=20)
axl.set_title('Normal Datasets', fontsize=20)
axr.set_title('Aleatorized Datasets', fontsize=20)
axl.set_xlabel('Best placed feature in the graphlet / single nodes', fontsize=16)
axr.set_xlabel('Best placed feature in the graphlet / single nodes', fontsize=16)
plt.show()

#### All Predictions

- Reading all predictions
- Obtaining the ranks of all nodes in each simulation
- See correlations and figures associating correlations

In [ ]:
# Reading Predictions
all_preds = {}
for i in range(20):
    with open(f'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets5Gap_All_Predictions_TAG_{i}.json', 'r') as file:
        all_preds[i] = json.load(file)
all_preds.keys()

In [ ]:
# Get the prediction changes into a dictionary
all_effect = {'Normal + Path': {}, 'Aleatorized + Path': {}}
for key in all_effect:
    for l in range(20):
        all_effect[key][l] = {}
        out_normal = pd.DataFrame(all_preds[l][key]['Normal'])
        for node in all_preds[l][key]:
            if node != 'Normal':
                all_effect[key][l][node] = pd.DataFrame(columns=all_preds[l][key][node].keys())
                for q in all_preds[l][key][node]:
                    out_shuffled = pd.DataFrame(all_preds[l][key][node][q])
                    results = pd.DataFrame((out_normal.values - out_shuffled.values)).abs()
                    all_effect[key][l][node][q] = results[0]

In [ ]:
# Get the prediction impact of nodes per simulation
pred_changes = {}
for key in all_effect:
    pred_changes[key] = {}
    for i in all_effect[key]:
        new_df = pd.DataFrame(columns=range(72))
        for node in all_effect[key][i].keys():
            new_df.loc[node] = all_effect[key][i][node].max(axis=1).values#.sort_values().mean()
        new_df = (new_df/new_df.sum()).replace({np.nan:0})
        pred_changes[key][i] = new_df
# Get closeness centrality
LGD_closeness = pd.Series(nx.closeness_centrality(LGD_FDiN))

In [ ]:
# Calculate all correlations within each simulation and each dataset
all_ranks_normal = pd.DataFrame()
all_ranks_aleat = pd.DataFrame()
closeness = LGD_closeness.loc[list(LGD_FDiN.nodes())]
correlations_normal = pd.DataFrame(columns=['All', 'Graphlets', 'Single Nodes'])
correlations_aleat = pd.DataFrame(columns=['All', 'Graphlets', 'Single Nodes'])
for iteration in pred_changes['Normal + Path']:
    curr_df = pred_changes['Normal + Path'][iteration].median(axis=1).rank(ascending=False).loc[list(LGD_FDiN.nodes())]
    all_ranks_normal = pd.concat((all_ranks_normal, curr_df))
    all_corr = stats.spearmanr(curr_df, closeness)[0]
    grap_corr = stats.spearmanr(curr_df.loc[graphlets[iteration]], closeness.loc[graphlets[iteration]])[0]
    sn_corr = stats.spearmanr(curr_df.loc[single_nodes[iteration]], closeness.loc[single_nodes[iteration]])[0]
    correlations_normal.loc[iteration] = [all_corr, grap_corr, sn_corr]

    curr_df = pred_changes['Aleatorized + Path'][iteration].median(axis=1).rank(
        ascending=False).loc[list(LGD_FDiN.nodes())]
    all_ranks_aleat = pd.concat((all_ranks_aleat, curr_df))
    all_corr = stats.spearmanr(curr_df, closeness)[0]
    grap_corr = stats.spearmanr(curr_df.loc[graphlets[iteration]], closeness.loc[graphlets[iteration]])[0]
    sn_corr = stats.spearmanr(curr_df.loc[single_nodes[iteration]], closeness.loc[single_nodes[iteration]])[0]
    correlations_aleat.loc[iteration] = [all_corr, grap_corr, sn_corr]
all_closenesses = pd.concat([closeness,]*20)

In [ ]:
colors = []
sizes = []
locator = []
for iteration in range(len(graphlets)):
    for i in LGD_FDiN.nodes():
        if i in graphlets[iteration]:
            locator.append('pathway')
            if i == gap_nodes[iteration]:
                colors.append('Black')
                sizes.append(8)
            else:
                colors.append('Red')
                sizes.append(5)
        elif i in single_nodes[iteration]:
            locator.append('single')
            colors.append('Green')
            sizes.append(5)
        else:
            locator.append('other')
            colors.append('skyblue')
            sizes.append(0.5)

In [ ]:
graphlet_corr = stats.spearmanr(all_ranks_normal.iloc[np.array(locator) == 'pathway'],
                                all_closenesses.iloc[np.array(locator) == 'pathway'])[0]

sn_corr = stats.spearmanr(all_ranks_normal.iloc[np.array(colors) == 'Green'],
                          all_closenesses.iloc[np.array(colors) == 'Green'])[0]

all_corr = stats.spearmanr(all_ranks_normal, all_closenesses)[0]

print('Original Dataset')
print(f'Graphlet     Correlation: {graphlet_corr}')
print(f'Single Nodes Correlation: {sn_corr}')
print(f'All          Correlation: {all_corr}')

In [ ]:
graphlet_corr = stats.spearmanr(all_ranks_aleat.iloc[np.array(locator) == 'pathway'],
                                all_closenesses.iloc[np.array(locator) == 'pathway'])[0]

sn_corr = stats.spearmanr(all_ranks_aleat.iloc[np.array(colors) == 'Green'],
                          all_closenesses.iloc[np.array(colors) == 'Green'])[0]

all_corr = stats.spearmanr(all_ranks_aleat, all_closenesses)[0]

print('Randomized Dataset')
print(f'Graphlet     Correlation: {graphlet_corr}')
print(f'Single Nodes Correlation: {sn_corr}')
print(f'All          Correlation: {all_corr}')

Correlations Figure

In [ ]:
fig, (axl,axr) = plt.subplots(1,2, figsize=(12,4), constrained_layout=True)
axl.scatter(all_ranks_normal, all_closenesses, s=sizes, c=colors)
axr.scatter(all_ranks_aleat, all_closenesses, s=sizes, c=colors)

**Gap Rank Distribution Figure**

**For Figure 7A.**

In [ ]:
gap_ranks.columns = ['RF - Original', 'PLS-DA - Original', 'FDiGNN - Original',
                    'RF - Randomized', 'PLS-DA - Randomized', 'FDiGNN - Randomized']
gap_ranks.iloc[:, :2] = gap_ranks.iloc[:, :2] / ds_size
gap_ranks.iloc[:, 2] = gap_ranks.iloc[:, 2] / gnn_size
gap_ranks.iloc[:, 3:5] = gap_ranks.iloc[:, 3:5] / ds_size
gap_ranks.iloc[:, 5] = gap_ranks.iloc[:, 5] / gnn_size

In [ ]:
f, (axl, axr) = plt.subplots(1,2, figsize=(4,4), constrained_layout=True)

bxp = {}
bxp_r = {}
for i in range(3):
    bxp[i] = axl.boxplot(gap_ranks.loc[:, [gap_ranks.columns[i]]],
                       positions=[i], manage_ticks=False, patch_artist=True,
                       medianprops={'color': 'black'}, widths=0.5)
    color = sns.color_palette('tab10', 10)[2]
    for patch in bxp[i]['boxes']:
        patch.set_facecolor(color)

    bxp_r[i] = axr.boxplot(gap_ranks.loc[:, [gap_ranks.columns[i+3]]],
                       positions=[i], manage_ticks=False, patch_artist=True,
                       medianprops={'color': 'black'}, widths=0.5)
    color = sns.color_palette('tab10', 10)[2]
    for patch in bxp_r[i]['boxes']:
        patch.set_facecolor(color)

axl.tick_params(axis='x', labelbottom=False,labeltop=True, labelsize=9.5)
axl.xaxis.set_ticks_position('top')
axl.set_xticks(range(3), ['RF', 'PLS-DA', 'FDiGNN'])
axl.set_ylim([1.03, -0.03])

axr.tick_params(axis='x', labelbottom=False,labeltop=True, labelsize=9.5)
axr.xaxis.set_ticks_position('top')
axr.set_xticks(range(3), ['RF', 'PLS-DA', 'FDiGNN'])
axr.set_ylim([1.03, -0.03])

axl.set_ylabel('Rank Percentile of Neutral Gap Nodes', fontsize=11)
axl.set_title('Original', fontsize=12)
axr.set_title('Randomized', fontsize=12)
f.suptitle('         LGD - 5 node graphlets with Neutral Gap', fontsize=12)
f.savefig('Paper_Figs/LGD_GapNodes_Graphlet5.png', dpi=600)
f.savefig('Paper_Figs/LGD_GapNodes_Graphlet5.svg', dpi=600)

## Graphlet Size 5 - OppGap

Reading and getting results

In [ ]:
# Parameters
size=5
ds_size = 1159
gnn_size = 350

colours = sns.color_palette('tab10', 20)

In [ ]:
# Obtaining results of each
graphlets = {}
for i in range(20):
    graphlets[i] = pd.read_excel(f'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets5OppGap_Normal_{i}.xlsx').to_dict()

with open(f'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets5_OppGap_TAG.json', 'w') as f:
    json.dump(graphlets, f)

In [ ]:
# Reading graphlets
with open('LGD_Simulations/Data/LGD_GNNPathwayTest_graphlets_size5.txt') as a:
    gs = a.read().split('\n')

with open('LGD_Simulations/Data/LGD_GNNPathwayTest_singlenodes_size5.txt') as a:
    sns_p = a.read().split('\n')

graphlets = []
for g in gs:
    graphlets.append(g.split(', '))

single_nodes = []
for g in sns_p:
    single_nodes.append(g.split(', '))

graphlets = graphlets[:-1]
single_nodes = single_nodes[:-1]

with open('LGD_Simulations/Data/LGD_5Graphlets_gaps.txt') as a:
    gap_nodes = a.read().split('\n')[:-1]

In [ ]:
graphlets_dict, graphlets_dfs, mannwhitney_pvalues, mannwhitney_pvalues_complete = process_results(
    'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets5_OppGap_TAG.json', size)

cols = []
for g in range(len(graphlets)):
    g_g = graphlets[g]
    special = gap_nodes[g]
    for node in g_g:
        if node == special:
            cols.append('Gap')
        else:
            cols.append('Pathway')
    cols.extend(['Single', 'Single','Single', 'Single', 'Single'])

Setting up Figures

**For Supplementary Figure 8A.**

In [ ]:
%%capture --no-display
fig, axs = plt.subplots(2,3, figsize=(16,8), constrained_layout=True)
gap_ranks = pd.DataFrame()
for ax, key in zip(axs.ravel(), ['RF - Normal', 'PLSDA - Normal', 'GNN - Normal - Pred',
                                'RF - Aleat.', 'PLSDA - Aleat.', 'GNN - Aleat. - Pred']):
    normal_ranks = graphlets_dfs[key].melt(ignore_index=False)
    normal_ranks['Type'] = cols
    normal_ranks.columns = ['DS', 'value', 'Type']
    normal_ranks['DS'] = 1 + normal_ranks['DS'].astype(int)
    gap_ranks[key] = normal_ranks[normal_ranks['Type'] == 'Gap']['value'].values
    sns.swarmplot(normal_ranks, x='DS', y='value', palette=colours, hue='Type', s=3,ax=ax,
                  hue_order=['Pathway', 'Single', 'Gap'], legend=False)
    ax.plot(range(len(graphlets)), graphlets_dfs[key].iloc[:size].median(), label='Pathway Nodes (Median)')
    ax.plot(range(len(graphlets)), graphlets_dfs[key].iloc[size:].median(), label='Single Nodes (Median)')
    ax.plot(range(len(graphlets)), normal_ranks[normal_ranks['Type'] == 'Gap']['value'].values,
            label='Gap Nodes')
    ax.set_ylim([50,0])
    ax.set_xlabel('')
    ax.set_ylabel('')
    #ax.set_ylabel('Importance Rank', fontsize=20)
    if 'PLSDA' in key:
        t1 = 'PLS-DA'
    elif 'GNN' in key:
        t1 = 'FDiGNN'
    else:
        t1 = 'RF'
    if 'Normal' in key:
        t2 = ' - Original'
    else:
        t2 = ' - Randomized'
        
    mw_P = f' - {mannwhitney_pvalues[key]:.3e}'
    a, b = mw_P.split("e")
        
    ax.set_title(t1 + t2 + f'{a}x$10^{{{b}}}$', fontsize=18)
    for i in mannwhitney_pvalues_complete.index:
        if mannwhitney_pvalues_complete.loc[i, key] < 0.05:
            ax.text(i, 58, '*', fontsize=20, horizontalalignment='center')
axs[0][0].set_ylabel('Importance Rank', fontsize=15)
axs[1][0].set_ylabel('Importance Rank', fontsize=15)
#axs[1][1].set_xlabel('Graphlet + Single Node Sets', fontsize=14)
axs[0][0].legend(fontsize=15)
fig.supxlabel('Graphlet + Single Node Sets', fontsize=18)
#axs[2][1].set_xlabel('Dataset', fontsize=14)
plt.show()
fig.savefig('Paper_Figs/LGD_Graphlets5_OppGap_TAG.png', dpi=600)
fig.savefig('Paper_Figs/LGD_Graphlets5_OppGap_TAG.svg', dpi=600)

In [ ]:
%%capture --no-display
fig, (axl, axr) = plt.subplots(1,2, figsize=(16,6), constrained_layout=True)
a=0
colours_plot = sns.color_palette('tab20', 20)
for key in graphlets_dfs.keys():
    if 'Aleat.' in key:
        ax = axr
        a -= 2
    else:
        ax = axl
    path_ranks = graphlets_dfs[key].iloc[:size].apply(lambda x: x.sort_values().values)
    single_ranks = graphlets_dfs[key].iloc[size:].apply(lambda x: x.sort_values().values)
    
    single_ranks_melt = single_ranks.T.copy()
    single_ranks_melt.columns = range(1,6)
    single_ranks_melt = single_ranks_melt.melt(ignore_index=True)
    single_ranks_melt['Type'] = ['Single',] * len(single_ranks_melt)
    path_ranks_melt = path_ranks.T.copy()
    path_ranks_melt.columns = range(1,size+1)
    path_ranks_melt = path_ranks_melt.melt(ignore_index=True)
    path_ranks_melt['Type'] = ['Pathway',] * len(path_ranks_melt)
    normal_ranks = graphlets_dfs[key].melt(ignore_index=False)
    normal_ranks['Type'] = cols
    normal_ranks.columns = ['DS', 'value', 'Type']
   # sns.swarmplot(pd.concat((path_ranks_melt, single_ranks_melt)),
    #              x='variable', y='value', palette=colours, hue='Type', s=3,ax=ax)
    
    ax.plot(range(1, size+1), path_ranks.median(axis=1), label=key + ' - Path', c=colours_plot[a])
    #ax.fill_between(range(1, size+1), path_ranks.min(axis=1), path_ranks.max(axis=1), color=colours_plot[a], alpha=0.1)
    ax.plot(range(1, 5+1), single_ranks.median(axis=1), label=key + ' - Single', c=colours_plot[a+1])
    ax.set_ylim([0,30])
    #ax.set_xlabel('')
    a += 2
axr.legend(fontsize=15, bbox_to_anchor=(1,1))
axl.set_ylabel('Rank', fontsize=20)
axl.set_title('Normal Datasets', fontsize=20)
axr.set_title('Aleatorized Datasets', fontsize=20)
axl.set_xlabel('Best placed feature in the graphlet / single nodes', fontsize=16)
axr.set_xlabel('Best placed feature in the graphlet / single nodes', fontsize=16)
plt.show()

#### All Predictions

- Reading all predictions
- Obtaining the ranks of all nodes in each simulation
- See correlations and figures associating correlations

In [ ]:
# Reading Predictions
all_preds = {}
for i in range(20):
    with open(f'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets5OppGap_All_Predictions_TAG_{i}.json', 'r') as file:
        all_preds[i] = json.load(file)
all_preds.keys()

In [ ]:
# Get the prediction changes into a dictionary
all_effect = {'Normal + Path': {}, 'Aleatorized + Path': {}}
for key in all_effect:
    for l in range(20):
        all_effect[key][l] = {}
        out_normal = pd.DataFrame(all_preds[l][key]['Normal'])
        for node in all_preds[l][key]:
            if node != 'Normal':
                all_effect[key][l][node] = pd.DataFrame(columns=all_preds[l][key][node].keys())
                for q in all_preds[l][key][node]:
                    out_shuffled = pd.DataFrame(all_preds[l][key][node][q])
                    results = pd.DataFrame((out_normal.values - out_shuffled.values)).abs()
                    all_effect[key][l][node][q] = results[0]

In [ ]:
# Get the prediction impact of nodes per simulation
pred_changes = {}
for key in all_effect:
    pred_changes[key] = {}
    for i in all_effect[key]:
        new_df = pd.DataFrame(columns=range(72))
        for node in all_effect[key][i].keys():
            new_df.loc[node] = all_effect[key][i][node].max(axis=1).values#.sort_values().mean()
        new_df = (new_df/new_df.sum()).replace({np.nan:0})
        pred_changes[key][i] = new_df
# Get closeness centrality
LGD_closeness = pd.Series(nx.closeness_centrality(LGD_FDiN))

In [ ]:
# Calculate all correlations within each simulation and each dataset
all_ranks_normal = pd.DataFrame()
all_ranks_aleat = pd.DataFrame()
closeness = LGD_closeness.loc[list(LGD_FDiN.nodes())]
correlations_normal = pd.DataFrame(columns=['All', 'Graphlets', 'Single Nodes'])
correlations_aleat = pd.DataFrame(columns=['All', 'Graphlets', 'Single Nodes'])
for iteration in pred_changes['Normal + Path']:
    curr_df = pred_changes['Normal + Path'][iteration].median(axis=1).rank(ascending=False).loc[list(LGD_FDiN.nodes())]
    all_ranks_normal = pd.concat((all_ranks_normal, curr_df))
    all_corr = stats.spearmanr(curr_df, closeness)[0]
    grap_corr = stats.spearmanr(curr_df.loc[graphlets[iteration]], closeness.loc[graphlets[iteration]])[0]
    sn_corr = stats.spearmanr(curr_df.loc[single_nodes[iteration]], closeness.loc[single_nodes[iteration]])[0]
    correlations_normal.loc[iteration] = [all_corr, grap_corr, sn_corr]

    curr_df = pred_changes['Aleatorized + Path'][iteration].median(axis=1).rank(
        ascending=False).loc[list(LGD_FDiN.nodes())]
    all_ranks_aleat = pd.concat((all_ranks_aleat, curr_df))
    all_corr = stats.spearmanr(curr_df, closeness)[0]
    grap_corr = stats.spearmanr(curr_df.loc[graphlets[iteration]], closeness.loc[graphlets[iteration]])[0]
    sn_corr = stats.spearmanr(curr_df.loc[single_nodes[iteration]], closeness.loc[single_nodes[iteration]])[0]
    correlations_aleat.loc[iteration] = [all_corr, grap_corr, sn_corr]
all_closenesses = pd.concat([closeness,]*20)

In [ ]:
colors = []
sizes = []
locator = []
for iteration in range(len(graphlets)):
    for i in LGD_FDiN.nodes():
        if i in graphlets[iteration]:
            locator.append('pathway')
            if i == gap_nodes[iteration]:
                colors.append('Black')
                sizes.append(8)
            else:
                colors.append('Red')
                sizes.append(5)
        elif i in single_nodes[iteration]:
            locator.append('single')
            colors.append('Green')
            sizes.append(5)
        else:
            locator.append('other')
            colors.append('skyblue')
            sizes.append(0.5)

In [ ]:
graphlet_corr = stats.spearmanr(all_ranks_normal.iloc[np.array(locator) == 'pathway'],
                                all_closenesses.iloc[np.array(locator) == 'pathway'])[0]

sn_corr = stats.spearmanr(all_ranks_normal.iloc[np.array(colors) == 'Green'],
                          all_closenesses.iloc[np.array(colors) == 'Green'])[0]

all_corr = stats.spearmanr(all_ranks_normal, all_closenesses)[0]

print('Original Dataset')
print(f'Graphlet     Correlation: {graphlet_corr}')
print(f'Single Nodes Correlation: {sn_corr}')
print(f'All          Correlation: {all_corr}')

In [ ]:
graphlet_corr = stats.spearmanr(all_ranks_aleat.iloc[np.array(locator) == 'pathway'],
                                all_closenesses.iloc[np.array(locator) == 'pathway'])[0]

sn_corr = stats.spearmanr(all_ranks_aleat.iloc[np.array(colors) == 'Green'],
                          all_closenesses.iloc[np.array(colors) == 'Green'])[0]

all_corr = stats.spearmanr(all_ranks_aleat, all_closenesses)[0]

print('Randomized Dataset')
print(f'Graphlet     Correlation: {graphlet_corr}')
print(f'Single Nodes Correlation: {sn_corr}')
print(f'All          Correlation: {all_corr}')

Correlations Figure

In [ ]:
fig, (axl,axr) = plt.subplots(1,2, figsize=(12,4), constrained_layout=True)
axl.scatter(all_ranks_normal, all_closenesses, s=sizes, c=colors)
axr.scatter(all_ranks_aleat, all_closenesses, s=sizes, c=colors)

**Gap Rank Distribution Figure**

**For Supplementary Figure 10D.**

In [ ]:
gap_ranks.columns = ['RF - Original', 'PLS-DA - Original', 'FDiGNN - Original',
                    'RF - Randomized', 'PLS-DA - Randomized', 'FDiGNN - Randomized']
gap_ranks.iloc[:, :2] = gap_ranks.iloc[:, :2] / ds_size
gap_ranks.iloc[:, 2] = gap_ranks.iloc[:, 2] / gnn_size
gap_ranks.iloc[:, 3:5] = gap_ranks.iloc[:, 3:5] / ds_size
gap_ranks.iloc[:, 5] = gap_ranks.iloc[:, 5] / gnn_size

In [ ]:
f, (axl, axr) = plt.subplots(1,2, figsize=(4,4), constrained_layout=True)

bxp = {}
bxp_r = {}
for i in range(3):
    bxp[i] = axl.boxplot(gap_ranks.loc[:, [gap_ranks.columns[i]]],
                       positions=[i], manage_ticks=False, patch_artist=True,
                       medianprops={'color': 'black'}, widths=0.5)
    color = sns.color_palette('tab10', 10)[2]
    for patch in bxp[i]['boxes']:
        patch.set_facecolor(color)

    bxp_r[i] = axr.boxplot(gap_ranks.loc[:, [gap_ranks.columns[i+3]]],
                       positions=[i], manage_ticks=False, patch_artist=True,
                       medianprops={'color': 'black'}, widths=0.5)
    color = sns.color_palette('tab10', 10)[2]
    for patch in bxp_r[i]['boxes']:
        patch.set_facecolor(color)

axl.tick_params(axis='x', labelbottom=False,labeltop=True, labelsize=9.5)
axl.xaxis.set_ticks_position('top')
axl.set_xticks(range(3), ['RF', 'PLS-DA', 'FDiGNN'])
axl.set_ylim([0.10, -0.001])

axr.tick_params(axis='x', labelbottom=False,labeltop=True, labelsize=9.5)
axr.xaxis.set_ticks_position('top')
axr.set_xticks(range(3), ['RF', 'PLS-DA', 'FDiGNN'])
axr.set_ylim([0.10, -0.001])

axl.set_ylabel('Rank Percentile of Opp. Gap Nodes', fontsize=11)
axl.set_title('Original', fontsize=12)
axr.set_title('Randomized', fontsize=12)
f.suptitle('         LGD - 5 node graphlets with Opp. Gap', fontsize=12)
f.savefig('Paper_Figs/LGD_OppGapNodes_Graphlet5.png', dpi=600)
f.savefig('Paper_Figs/LGD_OppGapNodes_Graphlet5.svg', dpi=600)

## Graphlet Size 6 - No Gap

Reading and getting results

In [ ]:
# Parameters
size=6
ds_size = 1159
gnn_size = 350

colours = sns.color_palette('tab10', 20)

In [ ]:
# Obtaining results of each
graphlets = {}
for i in range(20):
    graphlets[i] = pd.read_excel(f'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets6_Normal_{i}.xlsx').to_dict()

with open(f'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets6_Normal_TAG.json', 'w') as f:
    json.dump(graphlets, f)

In [ ]:
# Reading graphlets
with open('LGD_Simulations/Data/LGD_GNNPathwayTest_graphlets_size6.txt') as a:
    gs = a.read().split('\n')

with open('LGD_Simulations/Data/LGD_GNNPathwayTest_singlenodes_size6.txt') as a:
    sns_p = a.read().split('\n')

graphlets = []
for g in gs:
    graphlets.append(g.split(', '))

single_nodes = []
for g in sns_p:
    single_nodes.append(g.split(', '))

graphlets = graphlets[:-1]
single_nodes = single_nodes[:-1]

In [ ]:
graphlets_dict, graphlets_dfs, mannwhitney_pvalues, mannwhitney_pvalues_complete = process_results(
    'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets6_Normal_TAG.json', size)

cols = ['Pathway',] * size + ['Single',]*5
cols = cols*len(graphlets)

Setting up Figures

**For Supplementary Figure 5A.**

In [ ]:
%%capture --no-display
fig, axs = plt.subplots(2,3, figsize=(16,8), constrained_layout=True)
for ax, key in zip(axs.ravel(), ['RF - Normal', 'PLSDA - Normal', 'GNN - Normal - Pred',
                                'RF - Aleat.', 'PLSDA - Aleat.', 'GNN - Aleat. - Pred']):
    normal_ranks = graphlets_dfs[key].melt(ignore_index=False)
    normal_ranks['Type'] = cols
    normal_ranks.columns = ['DS', 'value', 'Type']
    normal_ranks['DS'] = 1 + normal_ranks['DS'].astype(int)
    sns.swarmplot(normal_ranks, x='DS', y='value', palette=colours, hue='Type', s=3,ax=ax, legend=False)
    ax.plot(range(len(graphlets)), graphlets_dfs[key].iloc[:size].median(), label='Pathway Nodes (Median)')
    ax.plot(range(len(graphlets)), graphlets_dfs[key].iloc[size:].median(), label='Single Nodes (Median)')
    ax.plot(range(len(graphlets)), graphlets_dfs[key].iloc[:size].max(), label='Pathway Nodes (Lowest)')
    ax.set_ylim([50,0])
    ax.set_xlabel('')
    ax.set_ylabel('')
    #ax.set_ylabel('Importance Rank', fontsize=20)
    if 'PLSDA' in key:
        t1 = 'PLS-DA'
    elif 'GNN' in key:
        t1 = 'FDiGNN'
    else:
        t1 = 'RF'
    if 'Normal' in key:
        t2 = ' - Original'
    else:
        t2 = ' - Randomized'
        
    mw_P = f' - {mannwhitney_pvalues[key]:.3e}'
    a, b = mw_P.split("e")
        
    ax.set_title(t1 + t2 + f'{a}x$10^{{{b}}}$', fontsize=18)
    for i in mannwhitney_pvalues_complete.index:
        if mannwhitney_pvalues_complete.loc[i, key] < 0.05:
            ax.text(i, 58, '*', fontsize=20, horizontalalignment='center')
axs[0][0].set_ylabel('Importance Rank', fontsize=15)
axs[1][0].set_ylabel('Importance Rank', fontsize=15)
#axs[1][1].set_xlabel('Graphlet + Single Node Sets', fontsize=14)
axs[0][0].legend(fontsize=15)
fig.supxlabel('Graphlet + Single Node Sets', fontsize=18)
#axs[2][1].set_xlabel('Dataset', fontsize=14)
plt.show()
fig.savefig('Paper_Figs/LGD_Graphlets6_TAG.png', dpi=600)
fig.savefig('Paper_Figs/LGD_Graphlets6_TAG.svg', dpi=600)

In [ ]:
%%capture --no-display
fig, (axl, axr) = plt.subplots(1,2, figsize=(16,6), constrained_layout=True)
a=0
colours_plot = sns.color_palette('tab20', 20)
for key in graphlets_dfs.keys():
    if 'Aleat.' in key:
        ax = axr
        a -= 2
    else:
        ax = axl
    path_ranks = graphlets_dfs[key].iloc[:size].apply(lambda x: x.sort_values().values)
    single_ranks = graphlets_dfs[key].iloc[size:].apply(lambda x: x.sort_values().values)
    
    single_ranks_melt = single_ranks.T.copy()
    single_ranks_melt.columns = range(1,6)
    single_ranks_melt = single_ranks_melt.melt(ignore_index=True)
    single_ranks_melt['Type'] = ['Single',] * len(single_ranks_melt)
    path_ranks_melt = path_ranks.T.copy()
    path_ranks_melt.columns = range(1,size+1)
    path_ranks_melt = path_ranks_melt.melt(ignore_index=True)
    path_ranks_melt['Type'] = ['Pathway',] * len(path_ranks_melt)
    normal_ranks = graphlets_dfs[key].melt(ignore_index=False)
    normal_ranks['Type'] = cols
    normal_ranks.columns = ['DS', 'value', 'Type']
   # sns.swarmplot(pd.concat((path_ranks_melt, single_ranks_melt)),
    #              x='variable', y='value', palette=colours, hue='Type', s=3,ax=ax)
    
    ax.plot(range(1, size+1), path_ranks.median(axis=1), label=key + ' - Path', c=colours_plot[a])
    #ax.fill_between(range(1, size+1), path_ranks.min(axis=1), path_ranks.max(axis=1), color=colours_plot[a], alpha=0.1)
    ax.plot(range(1, 5+1), single_ranks.median(axis=1), label=key + ' - Single', c=colours_plot[a+1])
    ax.set_ylim([0,30])
    #ax.set_xlabel('')
    a += 2
axr.legend(fontsize=15, bbox_to_anchor=(1,1))
axl.set_ylabel('Rank', fontsize=20)
axl.set_title('Normal Datasets', fontsize=20)
axr.set_title('Aleatorized Datasets', fontsize=20)
axl.set_xlabel('Best placed feature in the graphlet / single nodes', fontsize=16)
axr.set_xlabel('Best placed feature in the graphlet / single nodes', fontsize=16)
plt.show()

#### All Predictions

- Reading all predictions
- Obtaining the ranks of all nodes in each simulation
- See correlations and figures associating correlations

In [ ]:
# Reading Predictions
all_preds = {}
for i in range(20):
    with open(f'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets6_All_Predictions_TAG_{i}.json', 'r') as file:
        all_preds[i] = json.load(file)
all_preds.keys()

In [ ]:
# Get the prediction changes into a dictionary
all_effect = {'Normal + Path': {}, 'Aleatorized + Path': {}}
for key in all_effect:
    for l in range(20):
        all_effect[key][l] = {}
        out_normal = pd.DataFrame(all_preds[l][key]['Normal'])
        for node in all_preds[l][key]:
            if node != 'Normal':
                all_effect[key][l][node] = pd.DataFrame(columns=all_preds[l][key][node].keys())
                for q in all_preds[l][key][node]:
                    out_shuffled = pd.DataFrame(all_preds[l][key][node][q])
                    results = pd.DataFrame((out_normal.values - out_shuffled.values)).abs()
                    all_effect[key][l][node][q] = results[0]

In [ ]:
# Get the prediction impact of nodes per simulation
pred_changes = {}
for key in all_effect:
    pred_changes[key] = {}
    for i in all_effect[key]:
        new_df = pd.DataFrame(columns=range(72))
        for node in all_effect[key][i].keys():
            new_df.loc[node] = all_effect[key][i][node].max(axis=1).values#.sort_values().mean()
        new_df = (new_df/new_df.sum()).replace({np.nan:0})
        pred_changes[key][i] = new_df
# Get closeness centrality
LGD_closeness = pd.Series(nx.closeness_centrality(LGD_FDiN))

In [ ]:
# Calculate all correlations within each simulation and each dataset
all_ranks_normal = pd.DataFrame()
all_ranks_aleat = pd.DataFrame()
closeness = LGD_closeness.loc[list(LGD_FDiN.nodes())]
correlations_normal = pd.DataFrame(columns=['All', 'Graphlets', 'Single Nodes'])
correlations_aleat = pd.DataFrame(columns=['All', 'Graphlets', 'Single Nodes'])
for iteration in pred_changes['Normal + Path']:
    curr_df = pred_changes['Normal + Path'][iteration].median(axis=1).rank(ascending=False).loc[list(LGD_FDiN.nodes())]
    all_ranks_normal = pd.concat((all_ranks_normal, curr_df))
    all_corr = stats.spearmanr(curr_df, closeness)[0]
    grap_corr = stats.spearmanr(curr_df.loc[graphlets[iteration]], closeness.loc[graphlets[iteration]])[0]
    sn_corr = stats.spearmanr(curr_df.loc[single_nodes[iteration]], closeness.loc[single_nodes[iteration]])[0]
    correlations_normal.loc[iteration] = [all_corr, grap_corr, sn_corr]

    curr_df = pred_changes['Aleatorized + Path'][iteration].median(axis=1).rank(
        ascending=False).loc[list(LGD_FDiN.nodes())]
    all_ranks_aleat = pd.concat((all_ranks_aleat, curr_df))
    all_corr = stats.spearmanr(curr_df, closeness)[0]
    grap_corr = stats.spearmanr(curr_df.loc[graphlets[iteration]], closeness.loc[graphlets[iteration]])[0]
    sn_corr = stats.spearmanr(curr_df.loc[single_nodes[iteration]], closeness.loc[single_nodes[iteration]])[0]
    correlations_aleat.loc[iteration] = [all_corr, grap_corr, sn_corr]
all_closenesses = pd.concat([closeness,]*20)

In [ ]:
colors = []
sizes = []
for iteration in range(len(graphlets)):
    for i in LGD_FDiN.nodes():
        if i in graphlets[iteration]:
            colors.append('Red')
            sizes.append(5)
        elif i in single_nodes[iteration]:
            colors.append('Green')
            sizes.append(5)
        else:
            colors.append('skyblue')
            sizes.append(0.5)

In [ ]:
graphlet_corr = stats.spearmanr(all_ranks_normal.iloc[np.array(colors) == 'Red'],
                                all_closenesses.iloc[np.array(colors) == 'Red'])[0]

sn_corr = stats.spearmanr(all_ranks_normal.iloc[np.array(colors) == 'Green'],
                          all_closenesses.iloc[np.array(colors) == 'Green'])[0]

all_corr = stats.spearmanr(all_ranks_normal, all_closenesses)[0]

print('Original Dataset')
print(f'Graphlet     Correlation: {graphlet_corr}')
print(f'Single Nodes Correlation: {sn_corr}')
print(f'All          Correlation: {all_corr}')

In [ ]:
graphlet_corr = stats.spearmanr(all_ranks_aleat.iloc[np.array(colors) == 'Red'],
                                all_closenesses.iloc[np.array(colors) == 'Red'])[0]

sn_corr = stats.spearmanr(all_ranks_aleat.iloc[np.array(colors) == 'Green'],
                          all_closenesses.iloc[np.array(colors) == 'Green'])[0]

all_corr = stats.spearmanr(all_ranks_aleat, all_closenesses)[0]

print('Randomized Dataset')
print(f'Graphlet     Correlation: {graphlet_corr}')
print(f'Single Nodes Correlation: {sn_corr}')
print(f'All          Correlation: {all_corr}')

Correlations Figure

In [ ]:
fig, (axl,axr) = plt.subplots(1,2, figsize=(12,4), constrained_layout=True)
axl.scatter(all_ranks_normal, all_closenesses, s=sizes, c=colors)
axr.scatter(all_ranks_aleat, all_closenesses, s=sizes, c=colors)

In [ ]:
pred_changes['Aleatorized + Path'][9].median(axis=1).sort_values().tail(10)

## Graphlet Size 6 - Gap

Reading and getting results

In [ ]:
# Parameters
size=6
ds_size = 1159
gnn_size = 350

colours = sns.color_palette('tab10', 20)

In [ ]:
# Obtaining results of each
graphlets = {}
for i in range(20):
    graphlets[i] = pd.read_excel(f'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets6Gap_Normal_{i}.xlsx').to_dict()

with open(f'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets6_Gap_TAG.json', 'w') as f:
    json.dump(graphlets, f)

In [ ]:
# Reading graphlets
with open('LGD_Simulations/Data/LGD_GNNPathwayTest_graphlets_size6.txt') as a:
    gs = a.read().split('\n')

with open('LGD_Simulations/Data/LGD_GNNPathwayTest_singlenodes_size6.txt') as a:
    sns_p = a.read().split('\n')

graphlets = []
for g in gs:
    graphlets.append(g.split(', '))

single_nodes = []
for g in sns_p:
    single_nodes.append(g.split(', '))

graphlets = graphlets[:-1]
single_nodes = single_nodes[:-1]

with open('LGD_Simulations/Data/LGD_6Graphlets_gaps.txt') as a:
    gap_nodes = a.read().split('\n')[:-1]

In [ ]:
graphlets_dict, graphlets_dfs, mannwhitney_pvalues, mannwhitney_pvalues_complete = process_results(
    'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets6_Gap_TAG.json', size)

cols = []
for g in range(len(graphlets)):
    g_g = graphlets[g]
    special = gap_nodes[g]
    for node in g_g:
        if node == special:
            cols.append('Gap')
        else:
            cols.append('Pathway')
    cols.extend(['Single', 'Single','Single', 'Single', 'Single'])

Setting up Figures

**For Supplementary Figure 7A.**

In [ ]:
%%capture --no-display
fig, axs = plt.subplots(2,3, figsize=(16,8), constrained_layout=True)
gap_ranks = pd.DataFrame()
for ax, key in zip(axs.ravel(), ['RF - Normal', 'PLSDA - Normal', 'GNN - Normal - Pred',
                                'RF - Aleat.', 'PLSDA - Aleat.', 'GNN - Aleat. - Pred']):
    normal_ranks = graphlets_dfs[key].melt(ignore_index=False)
    normal_ranks['Type'] = cols
    normal_ranks.columns = ['DS', 'value', 'Type']
    normal_ranks['DS'] = 1 + normal_ranks['DS'].astype(int)
    gap_ranks[key] = normal_ranks[normal_ranks['Type'] == 'Gap']['value'].values
    sns.swarmplot(normal_ranks, x='DS', y='value', palette=colours, hue='Type', s=3,ax=ax,
                  hue_order=['Pathway', 'Single', 'Gap'], legend=False)
    ax.plot(range(len(graphlets)), graphlets_dfs[key].iloc[:size].median(), label='Pathway Nodes (Median)')
    ax.plot(range(len(graphlets)), graphlets_dfs[key].iloc[size:].median(), label='Single Nodes (Median)')
    ax.plot(range(len(graphlets)), normal_ranks[normal_ranks['Type'] == 'Gap']['value'].values,
            label='Gap Nodes')
    ax.set_ylim([50,0])
    ax.set_xlabel('')
    ax.set_ylabel('')
    #ax.set_ylabel('Importance Rank', fontsize=20)
    if 'PLSDA' in key:
        t1 = 'PLS-DA'
    elif 'GNN' in key:
        t1 = 'FDiGNN'
    else:
        t1 = 'RF'
    if 'Normal' in key:
        t2 = ' - Original'
    else:
        t2 = ' - Randomized'
        
    mw_P = f' - {mannwhitney_pvalues[key]:.3e}'
    a, b = mw_P.split("e")
        
    ax.set_title(t1 + t2 + f'{a}x$10^{{{b}}}$', fontsize=18)
    for i in mannwhitney_pvalues_complete.index:
        if mannwhitney_pvalues_complete.loc[i, key] < 0.05:
            ax.text(i, 58, '*', fontsize=20, horizontalalignment='center')
axs[0][0].set_ylabel('Importance Rank', fontsize=15)
axs[1][0].set_ylabel('Importance Rank', fontsize=15)
#axs[1][1].set_xlabel('Graphlet + Single Node Sets', fontsize=14)
axs[0][0].legend(fontsize=15)
fig.supxlabel('Graphlet + Single Node Sets', fontsize=18)
#axs[2][1].set_xlabel('Dataset', fontsize=14)
plt.show()
fig.savefig('Paper_Figs/LGD_Graphlets6_Gap_TAG.png', dpi=600)
fig.savefig('Paper_Figs/LGD_Graphlets6_Gap_TAG.svg', dpi=600)

In [ ]:
%%capture --no-display
fig, (axl, axr) = plt.subplots(1,2, figsize=(16,6), constrained_layout=True)
a=0
colours_plot = sns.color_palette('tab20', 20)
for key in graphlets_dfs.keys():
    if 'Aleat.' in key:
        ax = axr
        a -= 2
    else:
        ax = axl
    path_ranks = graphlets_dfs[key].iloc[:size].apply(lambda x: x.sort_values().values)
    single_ranks = graphlets_dfs[key].iloc[size:].apply(lambda x: x.sort_values().values)
    
    single_ranks_melt = single_ranks.T.copy()
    single_ranks_melt.columns = range(1,6)
    single_ranks_melt = single_ranks_melt.melt(ignore_index=True)
    single_ranks_melt['Type'] = ['Single',] * len(single_ranks_melt)
    path_ranks_melt = path_ranks.T.copy()
    path_ranks_melt.columns = range(1,size+1)
    path_ranks_melt = path_ranks_melt.melt(ignore_index=True)
    path_ranks_melt['Type'] = ['Pathway',] * len(path_ranks_melt)
    normal_ranks = graphlets_dfs[key].melt(ignore_index=False)
    normal_ranks['Type'] = cols
    normal_ranks.columns = ['DS', 'value', 'Type']
   # sns.swarmplot(pd.concat((path_ranks_melt, single_ranks_melt)),
    #              x='variable', y='value', palette=colours, hue='Type', s=3,ax=ax)
    
    ax.plot(range(1, size+1), path_ranks.median(axis=1), label=key + ' - Path', c=colours_plot[a])
    #ax.fill_between(range(1, size+1), path_ranks.min(axis=1), path_ranks.max(axis=1), color=colours_plot[a], alpha=0.1)
    ax.plot(range(1, 5+1), single_ranks.median(axis=1), label=key + ' - Single', c=colours_plot[a+1])
    ax.set_ylim([0,30])
    #ax.set_xlabel('')
    a += 2
axr.legend(fontsize=15, bbox_to_anchor=(1,1))
axl.set_ylabel('Rank', fontsize=20)
axl.set_title('Normal Datasets', fontsize=20)
axr.set_title('Aleatorized Datasets', fontsize=20)
axl.set_xlabel('Best placed feature in the graphlet / single nodes', fontsize=16)
axr.set_xlabel('Best placed feature in the graphlet / single nodes', fontsize=16)
plt.show()

#### All Predictions

- Reading all predictions
- Obtaining the ranks of all nodes in each simulation
- See correlations and figures associating correlations

In [ ]:
# Reading Predictions
all_preds = {}
for i in range(20):
    with open(f'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets6Gap_All_Predictions_TAG_{i}.json', 'r') as file:
        all_preds[i] = json.load(file)
all_preds.keys()

In [ ]:
# Get the prediction changes into a dictionary
all_effect = {'Normal + Path': {}, 'Aleatorized + Path': {}}
for key in all_effect:
    for l in range(20):
        all_effect[key][l] = {}
        out_normal = pd.DataFrame(all_preds[l][key]['Normal'])
        for node in all_preds[l][key]:
            if node != 'Normal':
                all_effect[key][l][node] = pd.DataFrame(columns=all_preds[l][key][node].keys())
                for q in all_preds[l][key][node]:
                    out_shuffled = pd.DataFrame(all_preds[l][key][node][q])
                    results = pd.DataFrame((out_normal.values - out_shuffled.values)).abs()
                    all_effect[key][l][node][q] = results[0]

In [ ]:
# Get the prediction impact of nodes per simulation
pred_changes = {}
for key in all_effect:
    pred_changes[key] = {}
    for i in all_effect[key]:
        new_df = pd.DataFrame(columns=range(72))
        for node in all_effect[key][i].keys():
            new_df.loc[node] = all_effect[key][i][node].max(axis=1).values#.sort_values().mean()
        new_df = (new_df/new_df.sum()).replace({np.nan:0})
        pred_changes[key][i] = new_df
# Get closeness centrality
LGD_closeness = pd.Series(nx.closeness_centrality(LGD_FDiN))

In [ ]:
# Calculate all correlations within each simulation and each dataset
all_ranks_normal = pd.DataFrame()
all_ranks_aleat = pd.DataFrame()
closeness = LGD_closeness.loc[list(LGD_FDiN.nodes())]
correlations_normal = pd.DataFrame(columns=['All', 'Graphlets', 'Single Nodes'])
correlations_aleat = pd.DataFrame(columns=['All', 'Graphlets', 'Single Nodes'])
for iteration in pred_changes['Normal + Path']:
    curr_df = pred_changes['Normal + Path'][iteration].median(axis=1).rank(ascending=False).loc[list(LGD_FDiN.nodes())]
    all_ranks_normal = pd.concat((all_ranks_normal, curr_df))
    all_corr = stats.spearmanr(curr_df, closeness)[0]
    grap_corr = stats.spearmanr(curr_df.loc[graphlets[iteration]], closeness.loc[graphlets[iteration]])[0]
    sn_corr = stats.spearmanr(curr_df.loc[single_nodes[iteration]], closeness.loc[single_nodes[iteration]])[0]
    correlations_normal.loc[iteration] = [all_corr, grap_corr, sn_corr]

    curr_df = pred_changes['Aleatorized + Path'][iteration].median(axis=1).rank(
        ascending=False).loc[list(LGD_FDiN.nodes())]
    all_ranks_aleat = pd.concat((all_ranks_aleat, curr_df))
    all_corr = stats.spearmanr(curr_df, closeness)[0]
    grap_corr = stats.spearmanr(curr_df.loc[graphlets[iteration]], closeness.loc[graphlets[iteration]])[0]
    sn_corr = stats.spearmanr(curr_df.loc[single_nodes[iteration]], closeness.loc[single_nodes[iteration]])[0]
    correlations_aleat.loc[iteration] = [all_corr, grap_corr, sn_corr]
all_closenesses = pd.concat([closeness,]*20)

In [ ]:
colors = []
sizes = []
locator = []
for iteration in range(len(graphlets)):
    for i in LGD_FDiN.nodes():
        if i in graphlets[iteration]:
            locator.append('pathway')
            if i == gap_nodes[iteration]:
                colors.append('Black')
                sizes.append(8)
            else:
                colors.append('Red')
                sizes.append(5)
        elif i in single_nodes[iteration]:
            locator.append('single')
            colors.append('Green')
            sizes.append(5)
        else:
            locator.append('other')
            colors.append('skyblue')
            sizes.append(0.5)

In [ ]:
graphlet_corr = stats.spearmanr(all_ranks_normal.iloc[np.array(locator) == 'pathway'],
                                all_closenesses.iloc[np.array(locator) == 'pathway'])[0]

sn_corr = stats.spearmanr(all_ranks_normal.iloc[np.array(colors) == 'Green'],
                          all_closenesses.iloc[np.array(colors) == 'Green'])[0]

all_corr = stats.spearmanr(all_ranks_normal, all_closenesses)[0]

print('Original Dataset')
print(f'Graphlet     Correlation: {graphlet_corr}')
print(f'Single Nodes Correlation: {sn_corr}')
print(f'All          Correlation: {all_corr}')

In [ ]:
graphlet_corr = stats.spearmanr(all_ranks_aleat.iloc[np.array(locator) == 'pathway'],
                                all_closenesses.iloc[np.array(locator) == 'pathway'])[0]

sn_corr = stats.spearmanr(all_ranks_aleat.iloc[np.array(colors) == 'Green'],
                          all_closenesses.iloc[np.array(colors) == 'Green'])[0]

all_corr = stats.spearmanr(all_ranks_aleat, all_closenesses)[0]

print('Randomized Dataset')
print(f'Graphlet     Correlation: {graphlet_corr}')
print(f'Single Nodes Correlation: {sn_corr}')
print(f'All          Correlation: {all_corr}')

Correlations Figure

In [ ]:
fig, (axl,axr) = plt.subplots(1,2, figsize=(12,4), constrained_layout=True)
axl.scatter(all_ranks_normal, all_closenesses, s=sizes, c=colors)
axr.scatter(all_ranks_aleat, all_closenesses, s=sizes, c=colors)

**Gap Rank Distribution Figure**

**For Supplementary Figure 10A.**

In [ ]:
gap_ranks.columns = ['RF - Original', 'PLS-DA - Original', 'FDiGNN - Original',
                    'RF - Randomized', 'PLS-DA - Randomized', 'FDiGNN - Randomized']
gap_ranks.iloc[:, :2] = gap_ranks.iloc[:, :2] / ds_size
gap_ranks.iloc[:, 2] = gap_ranks.iloc[:, 2] / gnn_size
gap_ranks.iloc[:, 3:5] = gap_ranks.iloc[:, 3:5] / ds_size
gap_ranks.iloc[:, 5] = gap_ranks.iloc[:, 5] / gnn_size

In [ ]:
f, (axl, axr) = plt.subplots(1,2, figsize=(4,4), constrained_layout=True)

bxp = {}
bxp_r = {}
for i in range(3):
    bxp[i] = axl.boxplot(gap_ranks.loc[:, [gap_ranks.columns[i]]],
                       positions=[i], manage_ticks=False, patch_artist=True,
                       medianprops={'color': 'black'}, widths=0.5)
    color = sns.color_palette('tab10', 10)[2]
    for patch in bxp[i]['boxes']:
        patch.set_facecolor(color)

    bxp_r[i] = axr.boxplot(gap_ranks.loc[:, [gap_ranks.columns[i+3]]],
                       positions=[i], manage_ticks=False, patch_artist=True,
                       medianprops={'color': 'black'}, widths=0.5)
    color = sns.color_palette('tab10', 10)[2]
    for patch in bxp_r[i]['boxes']:
        patch.set_facecolor(color)

axl.tick_params(axis='x', labelbottom=False,labeltop=True, labelsize=9.5)
axl.xaxis.set_ticks_position('top')
axl.set_xticks(range(3), ['RF', 'PLS-DA', 'FDiGNN'])
axl.set_ylim([1.03, -0.03])

axr.tick_params(axis='x', labelbottom=False,labeltop=True, labelsize=9.5)
axr.xaxis.set_ticks_position('top')
axr.set_xticks(range(3), ['RF', 'PLS-DA', 'FDiGNN'])
axr.set_ylim([1.03, -0.03])

axl.set_ylabel('Rank Percentile of Neutral Gap Nodes', fontsize=11)
axl.set_title('Original', fontsize=12)
axr.set_title('Randomized', fontsize=12)
f.suptitle('         LGD - 6 node graphlets with Neutral Gap', fontsize=12)
f.savefig('Paper_Figs/LGD_GapNodes_Graphlet6.png', dpi=600)
f.savefig('Paper_Figs/LGD_GapNodes_Graphlet6.svg', dpi=600)

## Graphlet Size 6 - OppGap

Reading and getting results

In [ ]:
# Parameters
size=6
ds_size = 1159
gnn_size = 350

colours = sns.color_palette('tab10', 20)

In [ ]:
# Obtaining results of each
graphlets = {}
for i in range(20):
    graphlets[i] = pd.read_excel(f'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets6OppGap_Normal_{i}.xlsx').to_dict()

with open(f'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets6_OppGap_TAG.json', 'w') as f:
    json.dump(graphlets, f)

In [ ]:
# Reading graphlets
with open('LGD_Simulations/Data/LGD_GNNPathwayTest_graphlets_size6.txt') as a:
    gs = a.read().split('\n')

with open('LGD_Simulations/Data/LGD_GNNPathwayTest_singlenodes_size6.txt') as a:
    sns_p = a.read().split('\n')

graphlets = []
for g in gs:
    graphlets.append(g.split(', '))

single_nodes = []
for g in sns_p:
    single_nodes.append(g.split(', '))

graphlets = graphlets[:-1]
single_nodes = single_nodes[:-1]

with open('LGD_Simulations/Data/LGD_6Graphlets_gaps.txt') as a:
    gap_nodes = a.read().split('\n')[:-1]

In [ ]:
graphlets_dict, graphlets_dfs, mannwhitney_pvalues, mannwhitney_pvalues_complete = process_results(
    'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets6_OppGap_TAG.json', size)

cols = []
for g in range(len(graphlets)):
    g_g = graphlets[g]
    special = gap_nodes[g]
    for node in g_g:
        if node == special:
            cols.append('Gap')
        else:
            cols.append('Pathway')
    cols.extend(['Single', 'Single','Single', 'Single', 'Single'])

Setting up Figures

**For Supplementary Figure 9A.**

In [ ]:
%%capture --no-display
fig, axs = plt.subplots(2,3, figsize=(16,8), constrained_layout=True)
gap_ranks = pd.DataFrame()
for ax, key in zip(axs.ravel(), ['RF - Normal', 'PLSDA - Normal', 'GNN - Normal - Pred',
                                'RF - Aleat.', 'PLSDA - Aleat.', 'GNN - Aleat. - Pred']):
    normal_ranks = graphlets_dfs[key].melt(ignore_index=False)
    normal_ranks['Type'] = cols
    normal_ranks.columns = ['DS', 'value', 'Type']
    normal_ranks['DS'] = 1 + normal_ranks['DS'].astype(int)
    gap_ranks[key] = normal_ranks[normal_ranks['Type'] == 'Gap']['value'].values
    sns.swarmplot(normal_ranks, x='DS', y='value', palette=colours, hue='Type', s=3,ax=ax,
                  hue_order=['Pathway', 'Single', 'Gap'], legend=False)
    ax.plot(range(len(graphlets)), graphlets_dfs[key].iloc[:size].median(), label='Pathway Nodes (Median)')
    ax.plot(range(len(graphlets)), graphlets_dfs[key].iloc[size:].median(), label='Single Nodes (Median)')
    ax.plot(range(len(graphlets)), normal_ranks[normal_ranks['Type'] == 'Gap']['value'].values,
            label='Gap Nodes')
    ax.set_ylim([50,0])
    ax.set_xlabel('')
    ax.set_ylabel('')
    #ax.set_ylabel('Importance Rank', fontsize=20)
    if 'PLSDA' in key:
        t1 = 'PLS-DA'
    elif 'GNN' in key:
        t1 = 'FDiGNN'
    else:
        t1 = 'RF'
    if 'Normal' in key:
        t2 = ' - Original'
    else:
        t2 = ' - Randomized'
        
    mw_P = f' - {mannwhitney_pvalues[key]:.3e}'
    a, b = mw_P.split("e")
        
    ax.set_title(t1 + t2 + f'{a}x$10^{{{b}}}$', fontsize=18)
    for i in mannwhitney_pvalues_complete.index:
        if mannwhitney_pvalues_complete.loc[i, key] < 0.05:
            ax.text(i, 58, '*', fontsize=20, horizontalalignment='center')
axs[0][0].set_ylabel('Importance Rank', fontsize=15)
axs[1][0].set_ylabel('Importance Rank', fontsize=15)
#axs[1][1].set_xlabel('Graphlet + Single Node Sets', fontsize=14)
axs[0][0].legend(fontsize=15)
fig.supxlabel('Graphlet + Single Node Sets', fontsize=18)
#axs[2][1].set_xlabel('Dataset', fontsize=14)
plt.show()
fig.savefig('Paper_Figs/LGD_Graphlets6_OppGap_TAG.png', dpi=600)
fig.savefig('Paper_Figs/LGD_Graphlets6_OppGap_TAG.svg', dpi=600)

In [ ]:
%%capture --no-display
fig, (axl, axr) = plt.subplots(1,2, figsize=(16,6), constrained_layout=True)
a=0
colours_plot = sns.color_palette('tab20', 20)
for key in graphlets_dfs.keys():
    if 'Aleat.' in key:
        ax = axr
        a -= 2
    else:
        ax = axl
    path_ranks = graphlets_dfs[key].iloc[:size].apply(lambda x: x.sort_values().values)
    single_ranks = graphlets_dfs[key].iloc[size:].apply(lambda x: x.sort_values().values)
    
    single_ranks_melt = single_ranks.T.copy()
    single_ranks_melt.columns = range(1,6)
    single_ranks_melt = single_ranks_melt.melt(ignore_index=True)
    single_ranks_melt['Type'] = ['Single',] * len(single_ranks_melt)
    path_ranks_melt = path_ranks.T.copy()
    path_ranks_melt.columns = range(1,size+1)
    path_ranks_melt = path_ranks_melt.melt(ignore_index=True)
    path_ranks_melt['Type'] = ['Pathway',] * len(path_ranks_melt)
    normal_ranks = graphlets_dfs[key].melt(ignore_index=False)
    normal_ranks['Type'] = cols
    normal_ranks.columns = ['DS', 'value', 'Type']
   # sns.swarmplot(pd.concat((path_ranks_melt, single_ranks_melt)),
    #              x='variable', y='value', palette=colours, hue='Type', s=3,ax=ax)
    
    ax.plot(range(1, size+1), path_ranks.median(axis=1), label=key + ' - Path', c=colours_plot[a])
    #ax.fill_between(range(1, size+1), path_ranks.min(axis=1), path_ranks.max(axis=1), color=colours_plot[a], alpha=0.1)
    ax.plot(range(1, 5+1), single_ranks.median(axis=1), label=key + ' - Single', c=colours_plot[a+1])
    ax.set_ylim([0,30])
    #ax.set_xlabel('')
    a += 2
axr.legend(fontsize=15, bbox_to_anchor=(1,1))
axl.set_ylabel('Rank', fontsize=20)
axl.set_title('Normal Datasets', fontsize=20)
axr.set_title('Aleatorized Datasets', fontsize=20)
axl.set_xlabel('Best placed feature in the graphlet / single nodes', fontsize=16)
axr.set_xlabel('Best placed feature in the graphlet / single nodes', fontsize=16)
plt.show()

#### All Predictions

- Reading all predictions
- Obtaining the ranks of all nodes in each simulation
- See correlations and figures associating correlations

In [ ]:
# Reading Predictions
all_preds = {}
for i in range(20):
    with open(f'LGD_Simulations/LGD_TAG_Results/LGD_Graphlets6OppGap_All_Predictions_TAG_{i}.json', 'r') as file:
        all_preds[i] = json.load(file)
all_preds.keys()

In [ ]:
# Get the prediction changes into a dictionary
all_effect = {'Normal + Path': {}, 'Aleatorized + Path': {}}
for key in all_effect:
    for l in range(20):
        all_effect[key][l] = {}
        out_normal = pd.DataFrame(all_preds[l][key]['Normal'])
        for node in all_preds[l][key]:
            if node != 'Normal':
                all_effect[key][l][node] = pd.DataFrame(columns=all_preds[l][key][node].keys())
                for q in all_preds[l][key][node]:
                    out_shuffled = pd.DataFrame(all_preds[l][key][node][q])
                    results = pd.DataFrame((out_normal.values - out_shuffled.values)).abs()
                    all_effect[key][l][node][q] = results[0]

In [ ]:
# Get the prediction impact of nodes per simulation
pred_changes = {}
for key in all_effect:
    pred_changes[key] = {}
    for i in all_effect[key]:
        new_df = pd.DataFrame(columns=range(72))
        for node in all_effect[key][i].keys():
            new_df.loc[node] = all_effect[key][i][node].max(axis=1).values#.sort_values().mean()
        new_df = (new_df/new_df.sum()).replace({np.nan:0})
        pred_changes[key][i] = new_df
# Get closeness centrality
LGD_closeness = pd.Series(nx.closeness_centrality(LGD_FDiN))

In [ ]:
# Calculate all correlations within each simulation and each dataset
all_ranks_normal = pd.DataFrame()
all_ranks_aleat = pd.DataFrame()
closeness = LGD_closeness.loc[list(LGD_FDiN.nodes())]
correlations_normal = pd.DataFrame(columns=['All', 'Graphlets', 'Single Nodes'])
correlations_aleat = pd.DataFrame(columns=['All', 'Graphlets', 'Single Nodes'])
for iteration in pred_changes['Normal + Path']:
    curr_df = pred_changes['Normal + Path'][iteration].median(axis=1).rank(ascending=False).loc[list(LGD_FDiN.nodes())]
    all_ranks_normal = pd.concat((all_ranks_normal, curr_df))
    all_corr = stats.spearmanr(curr_df, closeness)[0]
    grap_corr = stats.spearmanr(curr_df.loc[graphlets[iteration]], closeness.loc[graphlets[iteration]])[0]
    sn_corr = stats.spearmanr(curr_df.loc[single_nodes[iteration]], closeness.loc[single_nodes[iteration]])[0]
    correlations_normal.loc[iteration] = [all_corr, grap_corr, sn_corr]

    curr_df = pred_changes['Aleatorized + Path'][iteration].median(axis=1).rank(
        ascending=False).loc[list(LGD_FDiN.nodes())]
    all_ranks_aleat = pd.concat((all_ranks_aleat, curr_df))
    all_corr = stats.spearmanr(curr_df, closeness)[0]
    grap_corr = stats.spearmanr(curr_df.loc[graphlets[iteration]], closeness.loc[graphlets[iteration]])[0]
    sn_corr = stats.spearmanr(curr_df.loc[single_nodes[iteration]], closeness.loc[single_nodes[iteration]])[0]
    correlations_aleat.loc[iteration] = [all_corr, grap_corr, sn_corr]
all_closenesses = pd.concat([closeness,]*20)

In [ ]:
colors = []
sizes = []
locator = []
for iteration in range(len(graphlets)):
    for i in LGD_FDiN.nodes():
        if i in graphlets[iteration]:
            locator.append('pathway')
            if i == gap_nodes[iteration]:
                colors.append('Black')
                sizes.append(8)
            else:
                colors.append('Red')
                sizes.append(5)
        elif i in single_nodes[iteration]:
            locator.append('single')
            colors.append('Green')
            sizes.append(5)
        else:
            locator.append('other')
            colors.append('skyblue')
            sizes.append(0.5)

In [ ]:
graphlet_corr = stats.spearmanr(all_ranks_normal.iloc[np.array(locator) == 'pathway'],
                                all_closenesses.iloc[np.array(locator) == 'pathway'])[0]

sn_corr = stats.spearmanr(all_ranks_normal.iloc[np.array(colors) == 'Green'],
                          all_closenesses.iloc[np.array(colors) == 'Green'])[0]

all_corr = stats.spearmanr(all_ranks_normal, all_closenesses)[0]

print('Original Dataset')
print(f'Graphlet     Correlation: {graphlet_corr}')
print(f'Single Nodes Correlation: {sn_corr}')
print(f'All          Correlation: {all_corr}')

In [ ]:
graphlet_corr = stats.spearmanr(all_ranks_aleat.iloc[np.array(locator) == 'pathway'],
                                all_closenesses.iloc[np.array(locator) == 'pathway'])[0]

sn_corr = stats.spearmanr(all_ranks_aleat.iloc[np.array(colors) == 'Green'],
                          all_closenesses.iloc[np.array(colors) == 'Green'])[0]

all_corr = stats.spearmanr(all_ranks_aleat, all_closenesses)[0]

print('Randomized Dataset')
print(f'Graphlet     Correlation: {graphlet_corr}')
print(f'Single Nodes Correlation: {sn_corr}')
print(f'All          Correlation: {all_corr}')

Correlations Figure

In [ ]:
fig, (axl,axr) = plt.subplots(1,2, figsize=(12,4), constrained_layout=True)
axl.scatter(all_ranks_normal, all_closenesses, s=sizes, c=colors)
axr.scatter(all_ranks_aleat, all_closenesses, s=sizes, c=colors)

**Gap Rank Distribution Figure**

**For Supplementary Figure 10G.**

In [ ]:
gap_ranks.columns = ['RF - Original', 'PLS-DA - Original', 'FDiGNN - Original',
                    'RF - Randomized', 'PLS-DA - Randomized', 'FDiGNN - Randomized']
gap_ranks.iloc[:, :2] = gap_ranks.iloc[:, :2] / ds_size
gap_ranks.iloc[:, 2] = gap_ranks.iloc[:, 2] / gnn_size
gap_ranks.iloc[:, 3:5] = gap_ranks.iloc[:, 3:5] / ds_size
gap_ranks.iloc[:, 5] = gap_ranks.iloc[:, 5] / gnn_size

In [ ]:
f, (axl, axr) = plt.subplots(1,2, figsize=(4,4), constrained_layout=True)

bxp = {}
bxp_r = {}
for i in range(3):
    bxp[i] = axl.boxplot(gap_ranks.loc[:, [gap_ranks.columns[i]]],
                       positions=[i], manage_ticks=False, patch_artist=True,
                       medianprops={'color': 'black'}, widths=0.5)
    color = sns.color_palette('tab10', 10)[2]
    for patch in bxp[i]['boxes']:
        patch.set_facecolor(color)

    bxp_r[i] = axr.boxplot(gap_ranks.loc[:, [gap_ranks.columns[i+3]]],
                       positions=[i], manage_ticks=False, patch_artist=True,
                       medianprops={'color': 'black'}, widths=0.5)
    color = sns.color_palette('tab10', 10)[2]
    for patch in bxp_r[i]['boxes']:
        patch.set_facecolor(color)

axl.tick_params(axis='x', labelbottom=False,labeltop=True, labelsize=9.5)
axl.xaxis.set_ticks_position('top')
axl.set_xticks(range(3), ['RF', 'PLS-DA', 'FDiGNN'])
axl.set_ylim([0.10, -0.001])

axr.tick_params(axis='x', labelbottom=False,labeltop=True, labelsize=9.5)
axr.xaxis.set_ticks_position('top')
axr.set_xticks(range(3), ['RF', 'PLS-DA', 'FDiGNN'])
axr.set_ylim([0.10, -0.001])

axl.set_ylabel('Rank Percentile of Opp. Gap Nodes', fontsize=11)
axl.set_title('Original', fontsize=12)
axr.set_title('Randomized', fontsize=12)
f.suptitle('         LGD - 6 node graphlets with Opp. Gap', fontsize=12)
f.savefig('Paper_Figs/LGD_OppGapNodes_Graphlet6.png', dpi=600)
f.savefig('Paper_Figs/LGD_OppGapNodes_Graphlet6.svg', dpi=600)